# SenGuard-MSC kernel A: audit, preprocessing, baselines (G0, G1, G2)
Self-contained: downloads GEO data and clones author repositories itself. Runs notebooks 00, 01 and 02 in order and ends with frozen model artifacts.

In [ ]:
# Writes the shared library so this kernel needs no attached dataset.
from pathlib import Path
_src = Path('/kaggle/working/senguard_msc/src')
_src.mkdir(parents=True, exist_ok=True)
(_src / 'senguard.py').write_text('"""Shared code for SenGuard-MSC notebooks 01-05.\n\nConventions: expression frames are samples x genes. Fold-dependent fitting always happens on training\nsamples only. Nothing here reads the external MSC cohorts unless a caller passes them in.\n"""\nimport hashlib\nimport json\nimport os\nimport re\nimport sys\nimport warnings\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom joblib import Parallel, delayed\nfrom scipy import stats\nfrom sklearn.base import BaseEstimator, TransformerMixin, clone\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.feature_selection import SelectKBest, f_classif\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import average_precision_score, balanced_accuracy_score, brier_score_loss, f1_score, roc_auc_score\nfrom sklearn.model_selection import GridSearchCV, LeaveOneGroupOut\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.svm import SVC, LinearSVC\n\nSEED = 42\nDEV = \'GSE284666\'\nASC = \'GSE309666\'\nBM = \'GSE178514\'\n\n\ndef project_root():\n    if os.environ.get(\'SENGUARD_ROOT\'):\n        return Path(os.environ[\'SENGUARD_ROOT\'])\n    if Path(\'/kaggle/working/senguard_msc\').is_dir():\n        return Path(\'/kaggle/working/senguard_msc\')\n    cwd = Path.cwd()\n    return (cwd.parent if cwd.name == \'notebooks\' else cwd) / \'senguard_msc\'\n\n\nROOT = project_root()\nPROC, RESULTS, FIGS, REPORTS, MANIFESTS, EXT, CKPT = (ROOT / \'data/processed\', ROOT / \'results\', ROOT / \'figures\', ROOT / \'reports\',\n                                                      ROOT / \'manifests\', ROOT / \'external\', ROOT / \'checkpoints\')\n\n\ndef require(*paths):\n    missing = [str(p) for p in paths if not Path(p).exists()]\n    if missing:\n        raise FileNotFoundError(\'Required input missing; run the earlier notebook first:\\n  \' + \'\\n  \'.join(missing))\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for chunk in iter(lambda: f.read(1 << 20), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef output_manifest(paths):\n    rows = [{\'path\': str(p), \'bytes\': Path(p).stat().st_size, \'sha256\': sha256(p)} for p in paths]\n    return pd.DataFrame(rows)\n\n\n# ------------------------------------------------------------------------------------------- loading\ndef load_as_deposited(acc):\n    """Returns (expr samples x genes indexed by gene_id, annotation frame indexed by gene_id). No transformation."""\n    path = PROC / f\'{acc}_matrix_as_deposited.tsv.gz\'\n    require(path)\n    df = pd.read_csv(path, sep=\'\\t\', dtype={\'gene_id\': str, \'entrez\': str}, low_memory=False)\n    ann_cols = [c for c in df.columns if not c.startswith(\'GSM\')]\n    ann = df[ann_cols].set_index(\'gene_id\', drop=False)\n    expr = df.set_index(\'gene_id\')[[c for c in df.columns if c.startswith(\'GSM\')]].T\n    return expr.astype(float), ann\n\n\ndef load_manifest():\n    path = MANIFESTS / \'sample_manifest.csv\'\n    require(path)\n    return pd.read_csv(path, low_memory=False).set_index(\'sample_id\')\n\n\ndef dev_labels(manifest=None):\n    """SenCat labels follow the authors\' assemble_anndata.py: P and EV are controls, every other treatment is senescence-induced."""\n    m = load_manifest() if manifest is None else manifest\n    s = m[m.study_accession == DEV].copy()\n    s[\'y\'] = s.label_candidate.str.startswith(\'senescence_induced\').astype(int)\n    if s.label_candidate.eq(\'unclassified\').any():\n        raise ValueError(\'Unclassified SenCat treatment present\')\n    return s\n\n\ndef strip_version(ids):\n    return pd.Index(ids).str.replace(r\'\\.\\d+$\', \'\', regex=True)\n\n\n# ------------------------------------------------------------------------------------------- normalization\ndef mor_reference(counts):\n    """DESeq2 median-of-ratios reference: log geometric mean over genes non-zero in every reference sample."""\n    with np.errstate(divide=\'ignore\'):\n        logs = np.log(counts.to_numpy(dtype=float))\n    means = pd.Series(logs.mean(axis=0), index=counts.columns)\n    return means[np.isfinite(means)]\n\n\ndef mor_size_factors(counts, reference):\n    """Each sample\'s median log ratio to the reference. Zero counts in a new sample are excluded from its median."""\n    shared = counts.columns.intersection(reference.index)\n    x = counts[shared].to_numpy(dtype=float)\n    with np.errstate(divide=\'ignore\'):\n        logs = np.log(x)\n    ratios = logs - reference[shared].to_numpy()\n    ratios[~np.isfinite(ratios)] = np.nan\n    return pd.Series(np.exp(np.nanmedian(ratios, axis=1)), index=counts.index)\n\n\ndef author_gene_filter(counts):\n    """Filter in normalize_counts.py: non-zero in more than 5% of samples and total count above 380."""\n    x = counts.to_numpy(dtype=float)\n    keep = ((x != 0).sum(axis=0) > 0.05 * x.shape[0]) & (x.sum(axis=0) > 380)\n    return counts.loc[:, keep]\n\n\ndef mor_normalize(counts, reference=None):\n    ref = mor_reference(counts) if reference is None else reference\n    sf = mor_size_factors(counts, ref)\n    return counts.div(sf, axis=0), sf, ref\n\n\ndef tpm_from_fpkm(fpkm):\n    """Exact FPKM to TPM rescaling within the table. The denominator is whatever genes the table holds."""\n    return fpkm.div(fpkm.sum(axis=1), axis=0) * 1e6\n\n\ndef scale_to_million(x):\n    return x.div(x.sum(axis=1), axis=0) * 1e6\n\n\n# ------------------------------------------------------------------------------------------- scores\ndef load_sencat_markers():\n    path = EXT / \'wf-ml-markers-senescence/analysis/transcriptomics_loose5000f_tuned_common_features.csv\'\n    require(path)\n    return pd.read_csv(path).set_index(\'converted_alias\')\n\n\ndef load_husi_weights():\n    path = EXT / \'HUSI/sc/SenOCLR_features.csv\'\n    require(path)\n    return pd.read_csv(path).drop_duplicates(\'Gene\').set_index(\'Gene\')\n\n\ndef weighted_log1p_score(norm, coef):\n    """marker_classifier.py: log1p once, then X @ coef with no intercept; absent markers are skipped."""\n    shared = [g for g in coef.index if g in norm.columns]\n    return pd.Series(np.log1p(norm[shared].to_numpy(dtype=float)) @ coef.loc[shared].to_numpy(dtype=float), index=norm.index), len(shared)\n\n\ndef husi_raw(expr_by_symbol, weights):\n    """hUSI before the authors\' per-cohort min-max step: per-sample Spearman with the OCLR weight vector."""\n    shared = [g for g in weights.index if g in expr_by_symbol.columns]\n    w = stats.rankdata(weights.loc[shared, \'Weight\'].to_numpy())\n    x = expr_by_symbol[shared].to_numpy(dtype=float)\n    ranks = np.apply_along_axis(stats.rankdata, 1, x)\n    w_c, r_c = w - w.mean(), ranks - ranks.mean(axis=1, keepdims=True)\n    r = (r_c @ w_c) / (np.linalg.norm(r_c, axis=1) * np.linalg.norm(w_c))\n    return pd.Series(r, index=expr_by_symbol.index), len(shared)\n\n\ndef minmax(s):\n    return (s - s.min()) / (s.max() - s.min())\n\n\ndef read_gmt(path, names=None):\n    out = {}\n    with open(path, encoding=\'utf-8\', errors=\'replace\') as f:\n        for line in f:\n            parts = line.rstrip(\'\\n\').split(\'\\t\')\n            if len(parts) > 2 and (names is None or parts[0] in names):\n                out[parts[0]] = sorted({g for g in parts[2:] if g})\n    return out\n\n\ndef load_gene_sets():\n    h = read_gmt(EXT / \'HUSI/Data/h.all.v2023.1.Hs.symbols.gmt\')\n    extra = read_gmt(EXT / \'HUSI/Data/gene_50signatures_merge.gmt\', {\'SenMayo\', \'CellAge\', \'SASP\'})\n    return h, extra\n\n\ndef rank_set_score(expr_by_symbol, genes):\n    """Rank-based single-sample score, scaled to 0-1. Depends on one sample only, so no reference is fitted."""\n    shared = [g for g in genes if g in expr_by_symbol.columns]\n    if not shared:\n        return pd.Series(np.nan, index=expr_by_symbol.index), 0\n    ranks = expr_by_symbol.rank(axis=1, pct=True)\n    return ranks[shared].mean(axis=1), len(shared)\n\n\ndef to_symbols(expr, ann, sym_col=\'symbol\'):\n    """Collapses to symbols by keeping the highest-mean gene per symbol (recorded choice, annotation only)."""\n    sym = ann[sym_col].reindex(expr.columns)\n    keep = sym.notna() & (sym.astype(str) != \'nan\')\n    x = expr.loc[:, keep]\n    order = x.mean(axis=0).sort_values(ascending=False).index\n    s = sym.loc[order]\n    first = order[~s.duplicated().to_numpy()]\n    out = x[first].copy()\n    out.columns = sym.loc[first].to_numpy()\n    return out\n\n\n# ------------------------------------------------------------------------------------------- author pipeline\ndef author_feature_prefilter(norm, allowed_missing_samples=40, quantile=0.7):\n    x = norm.to_numpy(dtype=float)\n    keep = (x != 0).sum(axis=0) >= x.shape[0] - min(x.shape[0], allowed_missing_samples)\n    kept = norm.loc[:, keep]\n    totals = kept.sum(axis=0)\n    return kept.loc[:, totals > totals.quantile(quantile)]\n\n\ndef _author_fit_loco(xlog, y, groups, cols):\n    coefs = {}\n    for g in np.unique(groups):\n        tr = groups != g\n        m = LogisticRegression(solver=\'liblinear\').fit(xlog[cols].to_numpy()[tr], y[tr])\n        coefs[g] = m.coef_[0]\n    return coefs\n\n\ndef author_pipeline(norm, y, groups, k=5000, allowed_missing_samples=40, quantile=0.7):\n    """ml_classifier.py at config \'loose5000f\': selection on all samples given, then mean coefficient over leave-one-group-out fits."""\n    y, groups = np.asarray(y), np.asarray(groups)\n    pre = author_feature_prefilter(norm, allowed_missing_samples, quantile)\n    xlog = np.log1p(pre)\n    k = min(k, xlog.shape[1])\n    with warnings.catch_warnings():\n        warnings.simplefilter(\'ignore\')\n        sel = SelectKBest(f_classif, k=k).fit(xlog.to_numpy(), y)\n    cols = xlog.columns[sel.get_support(indices=True)]\n    coefs = _author_fit_loco(xlog, y, groups, cols)\n    return pd.Series(np.mean(list(coefs.values()), axis=0), index=cols, name=\'coef\'), coefs\n\n\n# ------------------------------------------------------------------------------------------- models\nclass ToArrayVariance(BaseEstimator, TransformerMixin):\n    """Drops zero-variance columns so f_classif stays defined inside a fold."""\n\n    def fit(self, X, y=None):\n        self.keep_ = np.asarray(X).std(axis=0) > 0\n        return self\n\n    def transform(self, X):\n        return np.asarray(X)[:, self.keep_]\n\n\ndef model_grid(xgb_device=\'cpu\'):\n    from xgboost import XGBClassifier\n    base = lambda clf, scale=True: Pipeline([(\'var\', ToArrayVariance()), (\'sel\', SelectKBest(f_classif)), (\'scale\', StandardScaler() if scale else \'passthrough\'), (\'clf\', clf)])\n    return {\n        \'logreg_l1\': (base(LogisticRegression(solver=\'liblinear\', penalty=\'l1\', max_iter=2000, random_state=SEED)),\n                      {\'sel__k\': [500, 2000], \'clf__C\': [0.01, 0.1, 1.0]}),\n        \'elastic_net\': (base(LogisticRegression(solver=\'saga\', penalty=\'elasticnet\', max_iter=3000, random_state=SEED)),\n                        {\'sel__k\': [2000], \'clf__C\': [0.01, 0.1], \'clf__l1_ratio\': [0.2, 0.8]}),\n        \'linear_svm\': (base(LinearSVC(max_iter=20000, random_state=SEED)), {\'sel__k\': [500, 2000], \'clf__C\': [0.0001, 0.001, 0.01]}),\n        \'rbf_svm\': (base(SVC(kernel=\'rbf\', gamma=\'scale\', random_state=SEED)), {\'sel__k\': [500, 2000], \'clf__C\': [1.0, 10.0]}),\n        \'random_forest\': (base(RandomForestClassifier(n_estimators=500, max_features=\'sqrt\', n_jobs=1, random_state=SEED), scale=False),\n                          {\'sel__k\': [2000], \'clf__min_samples_leaf\': [1, 3]}),\n        \'xgboost\': (base(XGBClassifier(n_estimators=200, learning_rate=0.05, subsample=0.8, colsample_bytree=0.5, tree_method=\'hist\', device=xgb_device,\n                                       n_jobs=1, random_state=SEED, eval_metric=\'logloss\'), scale=False),\n                    {\'sel__k\': [2000], \'clf__max_depth\': [2, 3]}),\n    }\n\n\ndef decision_values(model, X):\n    if hasattr(model, \'decision_function\'):\n        return model.decision_function(X)\n    return model.predict_proba(X)[:, 1]\n\n\ndef within_group_auc(estimator, X, y):\n    return roc_auc_score(y, decision_values(estimator, X))\n\n\ndef best_threshold(y, score):\n    """Threshold maximizing balanced accuracy; ties resolve to the lowest passing value\'s midpoint."""\n    order = np.sort(np.unique(score))\n    if len(order) == 1:\n        return float(order[0])\n    mids = (order[:-1] + order[1:]) / 2\n    ba = [balanced_accuracy_score(y, (score >= t).astype(int)) for t in mids]\n    return float(mids[int(np.argmax(ba))])\n\n\ndef prefilter_train(counts_train):\n    """Fold-local author filters and reference. Returns kept genes plus the reference fitted on the training fold."""\n    filtered = author_gene_filter(counts_train)\n    norm, _, ref = mor_normalize(filtered)\n    pre = author_feature_prefilter(norm)\n    return list(pre.columns), ref\n\n\ndef fold_transform(counts, genes, ref):\n    sf = mor_size_factors(counts[list(ref.index)], ref)\n    return np.log1p(counts[genes].div(sf, axis=0))\n\n\ndef run_outer_fold(name, pipe, grid, counts, y, groups, held):\n    """One leave-one-cell-type-out fold: normalization, filters, selection, tuning and threshold are all fit on the training cell types."""\n    tr = groups != held\n    with warnings.catch_warnings():\n        warnings.simplefilter(\'ignore\')\n        genes, ref = prefilter_train(counts.loc[tr])\n        Xtr, Xte = fold_transform(counts.loc[tr], genes, ref), fold_transform(counts.loc[~tr], genes, ref)\n        logo = list(LeaveOneGroupOut().split(Xtr, y[tr], groups[tr]))\n        gs = GridSearchCV(clone(pipe), grid, cv=logo, scoring=within_group_auc, n_jobs=1, refit=True, error_score=\'raise\')\n        gs.fit(Xtr.to_numpy(), y[tr])\n        oof = np.full(tr.sum(), np.nan)\n        for a, b in logo:\n            oof[b] = decision_values(clone(gs.best_estimator_).fit(Xtr.to_numpy()[a], y[tr][a]), Xtr.to_numpy()[b])\n        thr = best_threshold(y[tr], oof)\n        score = decision_values(gs.best_estimator_, Xte.to_numpy())\n    return {\'model\': name, \'held_out\': held, \'sample_id\': counts.index[~tr].tolist(), \'y\': y[~tr].tolist(), \'score\': score.tolist(), \'threshold\': thr,\n            \'oof_y\': y[tr].tolist(), \'oof_score\': oof.tolist(), \'best_params\': {k: (v.item() if hasattr(v, \'item\') else v) for k, v in gs.best_params_.items()},\n            \'inner_auc\': float(gs.best_score_), \'n_genes_prefilter\': len(genes)}\n\n\ndef run_nested_loco(counts, y, groups, grid, n_jobs=-1):\n    jobs = [(name, pipe, g, held) for name, (pipe, g) in grid.items() for held in np.unique(groups)]\n    return Parallel(n_jobs=n_jobs, verbose=0)(delayed(run_outer_fold)(n, p, g, counts, y, groups, h) for n, p, g, h in jobs)\n\n\n# ------------------------------------------------------------------------------------------- metrics\ndef fold_metrics(y, score, thr=None):\n    y, score = np.asarray(y), np.asarray(score)\n    out = {\'n\': len(y), \'n_pos\': int(y.sum()), \'auroc\': roc_auc_score(y, score), \'auprc\': average_precision_score(y, score)}\n    if thr is not None:\n        pred = (score >= thr).astype(int)\n        out.update(balanced_acc=balanced_accuracy_score(y, pred), macro_f1=f1_score(y, pred, average=\'macro\', zero_division=0))\n    return out\n\n\ndef bootstrap_mean_ci(values, n_boot=5000, seed=SEED):\n    """Percentile CI over cell types, the only independent-ish unit in the development set."""\n    v = np.asarray(values, dtype=float)\n    rng = np.random.default_rng(seed)\n    means = v[rng.integers(0, len(v), size=(n_boot, len(v)))].mean(axis=1)\n    return float(v.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))\n\n\ndef score_by_group_auc(score, y, groups):\n    rows = []\n    for g in np.unique(groups):\n        i = groups == g\n        rows.append({\'group\': g, **fold_metrics(y[i], score[i])})\n    return pd.DataFrame(rows)\n\n\n# ------------------------------------------------------------------------------------------- external trajectories\ndef slope_per_passage(passage, score):\n    return float(np.polyfit(np.asarray(passage, float), np.asarray(score, float), 1)[0])\n\n\ndef per_donor_trend(score, passage, donor):\n    """Per-donor slope per passage, Spearman, last minus first. Requires at least 3 passages for Spearman."""\n    df = pd.DataFrame({\'score\': score, \'passage\': passage, \'donor\': donor}).dropna()\n    rows = []\n    for d, g in df.groupby(\'donor\'):\n        g = g.sort_values(\'passage\')\n        rho = stats.spearmanr(g.passage, g.score).statistic if len(g) >= 3 else np.nan\n        rows.append({\'donor\': d, \'n_points\': len(g), \'first_passage\': int(g.passage.iloc[0]), \'last_passage\': int(g.passage.iloc[-1]),\n                     \'slope_per_passage\': slope_per_passage(g.passage, g.score), \'spearman_rho\': rho, \'last_minus_first\': g.score.iloc[-1] - g.score.iloc[0]})\n    return pd.DataFrame(rows)\n\n\ndef exact_sign_flip_p(values):\n    """Two-sided exact sign-flip test on the mean. With n donors the smallest reachable p is 2 / 2**n."""\n    v = np.asarray(values, float)\n    obs = abs(v.mean())\n    grid = np.array(np.meshgrid(*[[-1, 1]] * len(v))).reshape(len(v), -1).T\n    return float(np.mean(np.abs((grid * v).mean(axis=1)) >= obs - 1e-12))\n\n\ndef competitive_null(norm_log, coef, passage, donor, n=1000, seed=SEED, mode=\'weights\', universe=None):\n    """Mean per-donor slope for random signatures. \'weights\' shuffles weights among the same genes; \'genes\' draws random expressed genes."""\n    rng = np.random.default_rng(seed)\n    shared = [g for g in coef.index if g in norm_log.columns]\n    w = coef.loc[shared].to_numpy(dtype=float)\n    donors = np.asarray(donor)\n    p = np.asarray(passage, float)\n    groups = [np.flatnonzero(donors == d) for d in np.unique(donors)]\n    def mean_slope(mat, weights):\n        s = mat @ weights\n        return np.mean([np.polyfit(p[i], s[i], 1)[0] for i in groups])\n    x = norm_log[shared].to_numpy(dtype=float)\n    obs = mean_slope(x, w)\n    null = np.zeros(n)\n    if mode == \'weights\':\n        for i in range(n):\n            null[i] = mean_slope(x, rng.permutation(w))\n    else:\n        pool = list(norm_log.columns if universe is None else universe)\n        xp = norm_log[pool].to_numpy(dtype=float)\n        for i in range(n):\n            null[i] = mean_slope(xp[:, rng.choice(len(pool), len(shared), replace=False)], rng.permutation(w))\n    return float(obs), null, float((1 + np.sum(np.abs(null) >= abs(obs))) / (1 + n))\n\n\n# ------------------------------------------------------------------------------------------- figures\nPAL = {\'blue\': \'#2a78d6\', \'orange\': \'#eb6834\', \'aqua\': \'#1baf7a\', \'ink\': \'#0b0b0b\', \'ink2\': \'#52514e\', \'muted\': \'#8a897f\', \'grid\': \'#e6e5df\', \'surface\': \'#fcfcfb\'}\n\n\ndef set_fig_style():\n    import matplotlib as mpl\n    mpl.rcParams.update({\'figure.dpi\': 110, \'savefig.dpi\': 300, \'figure.facecolor\': PAL[\'surface\'], \'axes.facecolor\': PAL[\'surface\'], \'savefig.facecolor\': PAL[\'surface\'],\n                         \'axes.edgecolor\': PAL[\'muted\'], \'axes.linewidth\': 0.8, \'axes.grid\': True, \'grid.color\': PAL[\'grid\'], \'grid.linewidth\': 0.8, \'axes.axisbelow\': True,\n                         \'axes.spines.top\': False, \'axes.spines.right\': False, \'font.size\': 10, \'axes.titlesize\': 11, \'axes.titleweight\': \'bold\', \'axes.titlelocation\': \'left\',\n                         \'axes.labelcolor\': PAL[\'ink2\'], \'xtick.color\': PAL[\'ink2\'], \'ytick.color\': PAL[\'ink2\'], \'text.color\': PAL[\'ink\'], \'lines.linewidth\': 2,\n                         \'legend.frameon\': False, \'font.family\': \'DejaVu Sans\'})\n', encoding='utf-8')
print('senguard.py written', (_src / 'senguard.py').stat().st_size, 'bytes')

# # SenGuard-MSC · Notebook 00 — Environment and data audit (G0 + G1)

**Purpose:** verify the runtime (G0), then audit the three GEO cohorts (G1) *before* any label definition, normalization or model choice is locked. No senescence score is computed here, and no external MSC expression is used to choose anything.

## Where it runs
The same code runs locally or on Kaggle. Locally, run all cells from this folder; outputs go to `senguard_msc/` beside `notebooks/`, and files placed in `senguard_msc/input/` are used before any download. On Kaggle, follow the setup below.

## Kaggle setup (do this once)
1. Open https://www.kaggle.com/code → **New Notebook**.
2. **File → Import Notebook** → upload this `.ipynb`.
3. Right sidebar → **Session options**:
   - **Internet → On.** Kaggle requires a phone-verified account for this. If the toggle is unavailable, that is the one manual step only you can do.
   - **Accelerator:** `GPU` for the G0 hardware check, or `None` to save quota. G0/G1 need no GPU; a missing GPU is reported, not fatal.
   - **Environment:** *Always use latest environment* (needed for the preinstalled RAPIDS libraries).
4. **Run → Restart & Clear Cell Outputs**, then **Run All**. Read the last cell: it prints the gate status.
5. **Save Version → Save & Run All (Commit)**. Under *Advanced Settings* choose *Always save output*. Session files are lost when the session ends unless saved this way.
6. Send back: the last cell's printed report, and `reports/G1_DATA_AUDIT.md`.

Menu labels can move between Kaggle releases; the setting names above are what to look for.

## If NCBI file downloads fail on Kaggle
The downloader rejects empty or truncated files and the gate reports **BLOCKED** instead of continuing. To recover without using a laptop: Kaggle → **Datasets → New Dataset**, add the three file URLs printed by the failure message (remote URL option), then **Add Input** on this notebook and rerun. Any file with the exact GEO name found under `/kaggle/input` is used instead of downloading.

## Evidence tags used in every report
`[measured]` computed in this run · `[GEO]` GEO metadata text · `[published]` literature, machine-checked below · `[inference]` our reasoning, not stated by the source.

In [ ]:
# Config, paths, logging. Everything below derives from this cell; no hidden state.
import gzip, hashlib, html, io, json, logging, os, platform, re, shutil, subprocess, sys, tarfile, time
import urllib.parse, urllib.request
from datetime import datetime, timezone
from importlib import metadata as importlib_metadata
from pathlib import Path

import numpy as np
import pandas as pd
import requests

NOTEBOOK_ID = '00_environment_and_data_audit'
NOTEBOOK_VERSION = '1.0.0'
SEED = 42
np.random.seed(SEED)
RUN_UTC = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
ACCESS_DATE = RUN_UTC[:10]

IS_KAGGLE = Path('/kaggle/working').is_dir() and any(k.startswith('KAGGLE_') for k in os.environ)
RUNTIME = 'kaggle' if IS_KAGGLE else 'local'
_project = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ROOT = Path(os.environ.get('SENGUARD_ROOT', '/kaggle/working/senguard_msc' if IS_KAGGLE else _project / 'senguard_msc'))
INPUT_ROOT = Path(os.environ.get('SENGUARD_INPUT', '/kaggle/input' if IS_KAGGLE else ROOT / 'input'))

SUBDIRS = ['data/raw', 'data/metadata', 'data/processed', 'external', 'src', 'checkpoints',
           'results', 'figures', 'reports', 'logs', 'manifests']
for d in SUBDIRS:
    (ROOT / d).mkdir(parents=True, exist_ok=True)
RAW, META, PROC, EXT, REPORTS, LOGS, MANIFESTS = (ROOT / 'data/raw', ROOT / 'data/metadata', ROOT / 'data/processed',
                                                  ROOT / 'external', ROOT / 'reports', ROOT / 'logs', ROOT / 'manifests')

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s', force=True,
                    handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(LOGS / f'{NOTEBOOK_ID}.log', encoding='utf-8')])
log = logging.getLogger('senguard')

pd.set_option('display.width', 220, 'display.max_columns', 60, 'display.max_colwidth', 70, 'display.max_rows', 300)

ISSUES = []
def issue(gate, severity, dataset, message):
    """severity: BLOCKER (gate cannot pass), CONDITION (must be resolved or carried forward), NOTE."""
    ISSUES.append({'gate': gate, 'severity': severity, 'dataset': dataset, 'message': message})
    log.warning('[%s/%s/%s] %s', gate, severity, dataset, message)

def versioned_write(path, text):
    """Never destroy an earlier artifact: a changed file is kept beside the new one with a timestamp."""
    path = Path(path)
    if path.exists() and path.read_text(encoding='utf-8') != text:
        path.rename(path.with_name(f'{path.stem}.superseded_{datetime.now():%Y%m%d%H%M%S}{path.suffix}'))
    path.write_text(text, encoding='utf-8')
    return path

def save_csv(df, path):
    return versioned_write(path, df.to_csv(index=False))

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def md_table(df):
    cells = df.astype(str).apply(lambda col: col.str.replace('|', '/', regex=False).str.replace('\n', ' ', regex=False))
    rows = ['| ' + ' | '.join(map(str, df.columns)) + ' |', '|' + '---|' * len(df.columns)]
    rows += ['| ' + ' | '.join(r) + ' |' for r in cells.values.tolist()]
    return '\n'.join(rows)

print(f'{NOTEBOOK_ID} v{NOTEBOOK_VERSION} run={RUN_UTC} seed={SEED} runtime={RUNTIME}')
print('ROOT', ROOT, '| INPUT', INPUT_ROOT)

## G0 — Environment verification
Reports what is actually assigned. GPU model, VRAM and RAM are never assumed. The random tensors and matrix below are hardware smoke tests only and are never written to research outputs.

In [ ]:
def run(cmd, timeout=120, env=None):
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, env=env)
        return p.returncode, p.stdout.strip(), p.stderr.strip()
    except Exception as e:
        return -1, '', f'{type(e).__name__}: {e}'

env_report = {'notebook': NOTEBOOK_ID, 'notebook_version': NOTEBOOK_VERSION, 'run_utc': RUN_UTC, 'seed': SEED,
              'runtime': RUNTIME, 'kaggle_run_type': os.environ.get('KAGGLE_KERNEL_RUN_TYPE'),
              'python': sys.version.split()[0], 'platform': platform.platform()}

disk = shutil.disk_usage(ROOT)
env_report['disk_free_gb'] = round(disk.free / 1e9, 1)
env_report['disk_total_gb'] = round(disk.total / 1e9, 1)
if Path('/proc/meminfo').exists():
    mem = dict(l.split(':', 1) for l in Path('/proc/meminfo').read_text().splitlines())
    env_report['ram_total_gb'] = round(int(mem['MemTotal'].split()[0]) / 1e6, 1)
    env_report['ram_available_gb'] = round(int(mem['MemAvailable'].split()[0]) / 1e6, 1)
env_report['cpu_count'] = os.cpu_count()

working = Path('/kaggle/working')
env_report['input_listing'] = sorted(p.name for p in INPUT_ROOT.iterdir()) if INPUT_ROOT.is_dir() else []
env_report['working_listing'] = sorted(p.name for p in working.iterdir()) if working.is_dir() else []

versions = {}
for pkg in ['numpy', 'pandas', 'scipy', 'scikit-learn', 'requests', 'xgboost', 'scanpy', 'anndata', 'torch', 'cudf', 'cuml']:
    try:
        versions[pkg] = importlib_metadata.version(pkg)
    except importlib_metadata.PackageNotFoundError:
        versions[pkg] = None
env_report['packages'] = versions

code, out, err = run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'])
env_report['nvidia_smi'] = out.splitlines() if code == 0 else []
env_report['gpu_present'] = bool(env_report['nvidia_smi'])

smoke = {}
if env_report['gpu_present']:
    try:
        import torch
        smoke['torch_cuda_available'] = torch.cuda.is_available()
        smoke['torch_cuda_version'] = torch.version.cuda
        smoke['torch_device_count'] = torch.cuda.device_count()
        if torch.cuda.is_available():
            torch.manual_seed(SEED)
            smoke['torch_devices'] = [{'name': torch.cuda.get_device_name(i),
                                       'vram_gb': round(torch.cuda.get_device_properties(i).total_memory / 1e9, 2)}
                                      for i in range(torch.cuda.device_count())]
            a = torch.randn(2048, 2048, device='cuda')
            t = time.time(); b = (a @ a).sum().item(); torch.cuda.synchronize()
            smoke['torch_matmul_2048_s'] = round(time.time() - t, 4)
            smoke['torch_matmul_finite'] = bool(np.isfinite(b))
            del a
            torch.cuda.empty_cache()
    except Exception as e:
        smoke['torch_error'] = f'{type(e).__name__}: {e}'
    try:
        import xgboost as xgb
        rng = np.random.default_rng(SEED)
        Xs, ys = rng.normal(size=(2000, 50)), rng.integers(0, 2, 2000)
        t = time.time()
        xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=10, random_state=SEED).fit(Xs, ys)
        smoke['xgboost_cuda_fit_s'] = round(time.time() - t, 3)
    except Exception as e:
        smoke['xgboost_cuda_error'] = f'{type(e).__name__}: {e}'
    try:
        import cudf
        smoke['cudf_sum_check'] = cudf.DataFrame({'a': [1, 2, 3], 'b': [4, 5, 6]}).sum().to_pandas().to_dict()
    except Exception as e:
        smoke['cudf_error'] = f'{type(e).__name__}: {e}'
    try:
        import cuml
        smoke['cuml_import'] = cuml.__version__
    except Exception as e:
        smoke['cuml_error'] = f'{type(e).__name__}: {e}'
else:
    issue('G0', 'NOTE', 'runtime', 'No GPU assigned: CPU fallback. G1-G3 do not need one; only the optional G4 branch does.')
env_report['gpu_smoke_test'] = smoke
for key in [k for k in smoke if k.endswith('_error')]:
    issue('G0', 'NOTE', 'runtime', f'GPU stack: {key} = {smoke[key]}')

code, out, err = run([sys.executable, '-m', 'pip', 'freeze'], timeout=180)
if code == 0:
    (LOGS / 'pip_freeze.txt').write_text(out + '\n', encoding='utf-8')
print(json.dumps(env_report, indent=1, default=str))

In [ ]:
# Clone author repositories (shallow), record commit hashes. Their code is not executed here.
REPOS = {  # commit seen with `git ls-remote` when this notebook was written (2026-11-01); a change is reported, not fatal
    'wf-ml-markers-senescence': ('https://github.com/maragkakislab/wf-ml-markers-senescence', '0e23fcc926f31fc2206631ff9b64a852fb81fe89'),
    'HUSI': ('https://github.com/WJPina/HUSI', '300da32f111f8c8603e822eca8e84c9b5a76db48'),
    'bulkformer-latest': ('https://github.com/KangBoming/bulkformer-latest', '3a8407a7ab6b9b6c2f11929d5d53fb561123dc79'),
}
repo_records = []
git_env = dict(os.environ, GIT_LFS_SKIP_SMUDGE='1')  # never pull large weights in G0
for name, (url, authored_sha) in REPOS.items():
    dest = EXT / name
    rec = {'repo': name, 'url': url, 'sha_at_authoring': authored_sha, 'retrieved': ACCESS_DATE}
    if not (dest / '.git').is_dir():
        code, out, err = run(['git', 'clone', '--depth', '1', url, str(dest)], timeout=600, env=git_env)
        rec['clone_stderr_tail'] = err[-300:] if code else ''
    if (dest / '.git').is_dir():
        code, out, err = run(['git', '-C', str(dest), 'rev-parse', 'HEAD'])
        rec['sha'] = out if code == 0 else None
        rec['status'] = 'cloned'
        rec['license_files'] = [p.name for p in dest.iterdir() if p.name.upper().startswith(('LICENSE', 'COPYING'))]
        if rec['sha'] != authored_sha:
            issue('G0', 'NOTE', name, f"HEAD {rec['sha']} differs from the authored {authored_sha}; recorded for reproducibility.")
    else:
        rec['sha'], rec['status'] = None, 'not_cloned'
        issue('G0', 'CONDITION', name, f"Repository not cloned: {rec.get('clone_stderr_tail', '')}. Marker coverage needs it." )
    repo_records.append(rec)
env_report['repos'] = repo_records
print(pd.DataFrame(repo_records)[['repo', 'status', 'sha', 'license_files'] if any('license_files' in r for r in repo_records) else ['repo', 'status', 'sha']].to_string())

versioned_write(MANIFESTS / 'environment_report.json', json.dumps(env_report, indent=1, default=str))

blockers = [i for i in ISSUES if i['gate'] == 'G0' and i['severity'] == 'BLOCKER']
conditions = [i for i in ISSUES if i['gate'] == 'G0' and i['severity'] == 'CONDITION']
G0_STATUS = 'BLOCKED' if blockers else 'CONDITIONAL' if conditions else 'PASS'
print('G0 status:', G0_STATUS)

## G1 — Transport layer
What was seen when this notebook was written (2026-11-01, from outside Kaggle), so nothing here relies on a single route:
- GEO HTML pages can return a browser-verification page. The `form=text` endpoint returned real metadata.
- File routes sometimes returned valid headers and sizes but **0 body bytes**, and later worked. The cause is unknown and may not apply on Kaggle, so every file is validated (declared length, full gzip CRC or tar walk) before it counts as present, and each route is retried.

In [ ]:
UA = {'User-Agent': 'SenGuard-MSC-audit/1.0 (research notebook)'}
GEO_FTP = 'https://ftp.ncbi.nlm.nih.gov/geo'
GEO_CGI = 'https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi'
EUTILS = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils'
MAX_DOWNLOAD_BYTES = 200_000_000  # processed tables only; nothing here should approach this

def series_dir(acc):
    return f'{GEO_FTP}/series/{acc[:-3]}nnn/{acc}'

def http_get(url, tries=4, timeout=120, **kw):
    delay = 3
    last = None
    for attempt in range(tries):
        try:
            r = requests.get(url, headers=UA, timeout=timeout, **kw)
            if r.status_code in (403, 429, 500, 502, 503, 504):
                raise requests.HTTPError(f'HTTP {r.status_code}')
            r.raise_for_status()
            time.sleep(0.4)
            return r
        except Exception as e:
            last = e
            time.sleep(delay)
            delay *= 2
    raise RuntimeError(f'GET failed after {tries} tries: {url}: {last}')

def http_head(url):
    for attempt in range(3):
        try:
            r = requests.head(url, headers=UA, timeout=60, allow_redirects=True)
            if r.status_code == 200:
                time.sleep(0.4)
                return {'status': 200, 'bytes': int(r.headers.get('Content-Length', 0)) or None,
                        'last_modified': r.headers.get('Last-Modified')}
            status = r.status_code
        except Exception as e:
            status = f'{type(e).__name__}'
        time.sleep(3 * (attempt + 1))
    return {'status': status, 'bytes': None, 'last_modified': None}

def validate_file(path, kind, expected_bytes=None):
    """Returns None when valid, else the reason."""
    path = Path(path)
    if not path.is_file() or path.stat().st_size == 0:
        return 'empty or missing'
    size = path.stat().st_size
    if expected_bytes and size != expected_bytes:
        return f'size {size} != declared {expected_bytes}'
    try:
        if kind == 'gz':
            with gzip.open(path, 'rb') as f:
                while f.read(1 << 20):
                    pass
        elif kind == 'tar':
            with tarfile.open(path, 'r:*') as tf:
                if not tf.getmembers():
                    return 'tar has no members'
    except Exception as e:
        return f'{kind} check failed: {type(e).__name__}: {e}'
    return None

_attached_index = None
def find_attached(filename):
    global _attached_index
    if _attached_index is None:
        _attached_index = {}
        if INPUT_ROOT.is_dir():
            for p in INPUT_ROOT.rglob('*'):
                if p.is_file():
                    _attached_index.setdefault(p.name, p)
    return _attached_index.get(filename)

def _via_requests(url, part):
    with requests.get(url, headers=UA, timeout=300, stream=True) as r:
        r.raise_for_status()
        with open(part, 'wb') as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)

def _via_curl(url, part):
    code, _, err = run(['curl', '-fsSL', '--http1.1', '-A', UA['User-Agent'], '-o', str(part), url], timeout=600)
    if code != 0:
        raise RuntimeError(err[-200:])

def _via_ftp(url, part):
    ftp_url = url.replace('https://ftp.ncbi.nlm.nih.gov', 'ftp://ftp.ncbi.nlm.nih.gov')
    with urllib.request.urlopen(ftp_url, timeout=300) as r, open(part, 'wb') as f:
        shutil.copyfileobj(r, f)

def fetch_file(url, dest, kind, expected_bytes=None):
    """Attached Kaggle input → cached working copy → requests → curl → FTP. A file only counts once validated."""
    dest = Path(dest)
    rec = {'file': dest.name, 'url': url, 'declared_bytes': expected_bytes, 'route': None, 'path': None,
           'bytes': None, 'sha256': None, 'attempts': [], 'retrieved': ACCESS_DATE}
    attached = find_attached(dest.name)
    if attached is not None:
        why = validate_file(attached, kind, None)
        rec['attempts'].append(f'attached:{why or "ok"}')
        if why is None:
            rec.update(route='kaggle_input', path=str(attached), bytes=attached.stat().st_size, sha256=sha256(attached))
            return rec
    if dest.exists():
        why = validate_file(dest, kind, expected_bytes)
        rec['attempts'].append(f'cache:{why or "ok"}')
        if why is None:
            rec.update(route='cached', path=str(dest), bytes=dest.stat().st_size, sha256=sha256(dest))
            return rec
    part = dest.with_name(dest.name + '.part')
    for route, fn in [('https', _via_requests), ('curl', _via_curl), ('ftp', _via_ftp)]:
        for attempt in range(2):
            try:
                part.unlink(missing_ok=True)
                fn(url, part)
                why = validate_file(part, kind, expected_bytes)
            except Exception as e:
                why = f'{type(e).__name__}: {str(e)[:150]}'
            rec['attempts'].append(f'{route}:{why or "ok"}')
            if why is None:
                os.replace(part, dest)
                rec.update(route=route, path=str(dest), bytes=dest.stat().st_size, sha256=sha256(dest))
                return rec
            time.sleep(4 * (attempt + 1))
    part.unlink(missing_ok=True)
    return rec

## G1.1 — GEO metadata (family SOFT, then `form=text`)

In [ ]:
ACCESSIONS = ['GSE284666', 'GSE309666', 'GSE178514']
# What the sample listings showed when this notebook was written. A mismatch is flagged; nothing is forced to match.
EXPECTED_N = {'GSE284666': 190, 'GSE309666': 18, 'GSE178514': 18}

def parse_geo_text(text):
    series, samples, cur = {}, [], None
    for line in text.splitlines():
        if line.startswith('^'):
            key, _, val = line[1:].partition('=')
            key, val = key.strip().upper(), val.strip()
            if key == 'SAMPLE':
                cur = {'gsm': [val]}
                samples.append(cur)
            elif key == 'SERIES':
                cur = series
                cur['accession'] = [val]
            else:
                cur = None
        elif line.startswith('!') and cur is not None and ' = ' in line:
            key, _, val = line[1:].partition(' = ')
            cur.setdefault(key.strip(), []).append(val.strip())
    return series, samples

def looks_like_geo(text):
    return '^SAMPLE' in text or '^SERIES' in text

def get_metadata(acc):
    """Returns (series, samples, route, saved_path). Raises if no route gives real GEO text."""
    errors = []
    soft = f'{acc}_family.soft.gz'
    rec = fetch_file(f'{series_dir(acc)}/soft/{soft}', META / soft, 'gz')
    if rec['path']:
        text = gzip.open(rec['path'], 'rt', encoding='utf-8', errors='replace').read()
        series, samples = parse_geo_text(text)
        if samples and series:
            return series, samples, f"family_soft:{rec['route']}", rec['path']
        errors.append('SOFT parsed to nothing')
    errors.append(f"SOFT: {rec['attempts'][-1] if rec['attempts'] else 'no attempt'}")
    self_txt = http_get(GEO_CGI, params={'acc': acc, 'targ': 'self', 'form': 'text', 'view': 'full'}).text
    gsm_txt = http_get(GEO_CGI, params={'acc': acc, 'targ': 'gsm', 'form': 'text', 'view': 'full'}).text
    if not (looks_like_geo(self_txt) and looks_like_geo(gsm_txt)):
        raise RuntimeError(f'No GEO text returned ({errors}); the response may be a browser-verification page.')
    series, _ = parse_geo_text(self_txt)
    _, samples = parse_geo_text(gsm_txt)
    path = META / f'{acc}_geo_form_text.txt'
    versioned_write(path, self_txt + '\n' + gsm_txt)
    return series, samples, 'geo_form_text', str(path)

GEO = {}
meta_rows = []
for acc in ACCESSIONS:
    try:
        series, samples, route, path = get_metadata(acc)
        GEO[acc] = {'series': series, 'samples': samples}
        listed = len(series.get('Series_sample_id', []))
        meta_rows.append({'accession': acc, 'route': route, 'path': path, 'samples_parsed': len(samples),
                          'samples_listed_in_series': listed, 'expected_at_authoring': EXPECTED_N[acc],
                          'title': series.get('Series_title', [''])[0], 'pubmed': ';'.join(series.get('Series_pubmed_id', [])),
                          'public': series.get('Series_status', [''])[0], 'retrieved': ACCESS_DATE})
        if len(samples) != listed or len(samples) != EXPECTED_N[acc]:
            issue('G1', 'CONDITION', acc, f'{len(samples)} parsed, {listed} listed, {EXPECTED_N[acc]} expected.')
    except Exception as e:
        issue('G1', 'BLOCKER', acc, f'Metadata unavailable: {e}')
meta_table = pd.DataFrame(meta_rows)
save_csv(meta_table, MANIFESTS / 'geo_metadata_sources.csv')
if len(meta_table):
    print(meta_table.drop(columns=['path']).to_string())

## G1.2 — Sample manifest
Fields are parsed per study from GEO text. `label_candidate` records what the deposit supports; it is **not** a locked analysis label. Passage is never converted into a senescence label.

In [ ]:
def chars(s):
    out = {}
    for c in s.get('Sample_characteristics_ch1', []):
        k, _, v = c.partition(':')
        out[k.strip().lower()] = v.strip()
    return out

def first(s, key, default=''):
    return s.get(key, [default])[0]

def base_row(acc, s):
    ch = chars(s)
    supp = [v.rsplit('/', 1)[-1] for k, vals in s.items() if k.startswith('Sample_supplementary_file') for v in vals if v != 'NONE']
    gsm = first(s, 'gsm')
    return {'sample_id': gsm, 'study_accession': acc, 'title': first(s, 'Sample_title'),
            'source': first(s, 'Sample_source_name_ch1'), 'cell_type': ch.get('cell type', ''), 'cell_line': ch.get('cell line', ''),
            'donor': '', 'donor_evidence': '', 'treatment': ch.get('treatment', ''), 'passage': np.nan, 'passage_evidence': '',
            'replicate': '', 'technology': f"{first(s, 'Sample_instrument_model')} ({first(s, 'Sample_platform_id')})",
            'assay': first(s, 'Sample_library_strategy'), 'library_selection': first(s, 'Sample_library_selection'),
            'molecule': first(s, 'Sample_molecule_ch1'), 'timestamp': '',
            'geo_submission_date': first(s, 'Sample_submission_date'), 'sample_supp_files': ';'.join(supp),
            'library_name': next((d.split(':', 1)[1].strip() for d in s.get('Sample_description', []) if d.lower().startswith('library name')), ''),
            'available_labels': '', 'label_candidate': '', 'label_evidence': '', 'use_in_mrna_analysis': False,
            'source_link': f'https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc={gsm}', 'evidence_notes': ''}

def sencat_condition(treatment):
    t = treatment.lower()
    if t.startswith('control'):
        return 'control_proliferating', '[GEO] treatment field'
    if t.startswith('empty vector'):
        return 'control_empty_vector', '[GEO] treatment field; vector control for RasV12'
    if 'senescence' in t:
        return 'senescence_induced', '[GEO] treatment field names senescence; paradigm-level, not per-sample, confirmation'
    if 'rasv12' in t:
        return 'senescence_induced_oncogene', '[inference] RasV12 treatment; the field itself does not say senescence'
    return 'unclassified', 'treatment field not recognized'

def sencat_row(acc, s):
    r = base_row(acc, s)
    cond, ev = sencat_condition(r['treatment'])
    m = re.search(r'_(\d+)$', r['title'])
    r.update(replicate=m.group(1) if m else '', available_labels='treatment condition; cell line',
             donor='not provided', donor_evidence='[GEO] no donor field; replicates sit within a cell_line and are not donors',
             label_candidate=cond, label_evidence=ev, use_in_mrna_analysis=True,
             evidence_notes='Grouping unit is cell_line. Induced-senescence condition; dose and duration in treatment field.')
    return r

def asc_row(acc, s):
    r = base_row(acc, s)
    ch = chars(s)
    donor = ch.get('donor', '').replace('Donor', '').strip()
    pm = re.search(r'p(\d+)', ch.get('passage', ''), flags=re.I)
    r.update(donor=donor, donor_evidence='[GEO] donor field', passage=int(pm.group(1)) if pm else np.nan,
             passage_evidence='[GEO] passage field', available_labels='donor; passage',
             label_candidate='none_passage_only', label_evidence='No assay-confirmed per-sample senescence label in the deposit',
             use_in_mrna_analysis=first(s, 'Sample_library_strategy') == 'RNA-Seq',
             evidence_notes='Growth-arrest status is published per donor at final passage only; see G1.5.')
    return r

def bm_row(acc, s):
    r = base_row(acc, s)
    ch = chars(s)
    t = r['title']
    tm = re.search(r'P(\d+)_REP(\d+)', t)
    bm = re.search(r'\[P(\d+)(\d{3})\]', t)
    r['passage'] = int(tm.group(1)) if tm else np.nan
    r['replicate'] = tm.group(2) if tm else ''
    r['candidate_biological_id'] = bm.group(2) if bm else ''
    r['generation_field'] = ch.get('generation', '')
    gen = re.search(r'(\d+)', r['generation_field'])
    r['generation_conflicts_passage'] = bool(gen and tm and int(gen.group(1)) != int(tm.group(1)))
    r.update(donor='not stated', donor_evidence='[inference] title suffix recurs across passages; meaning not stated; tested in G1.6',
             passage_evidence='[GEO] title and treatment field; generation field not used, see G1.5',
             available_labels='passage', label_candidate='none_passage_only',
             label_evidence='No assay-confirmed per-sample senescence label in the deposit',
             use_in_mrna_analysis=first(s, 'Sample_library_strategy') == 'RNA-Seq',
             evidence_notes='miRNA-Seq record: excluded from mRNA matrix.' if first(s, 'Sample_library_strategy') != 'RNA-Seq' else 'mRNA record.')
    return r

ROW_BUILDERS = {'GSE284666': sencat_row, 'GSE309666': asc_row, 'GSE178514': bm_row}
manifest_parts = [pd.DataFrame([ROW_BUILDERS[a](a, s) for s in GEO[a]['samples']]) for a in ACCESSIONS if a in GEO]
manifest = pd.concat(manifest_parts, ignore_index=True) if manifest_parts else pd.DataFrame()

if len(manifest):
    print(manifest.groupby(['study_accession', 'assay']).size().to_string())
    unclassified = manifest[manifest.label_candidate == 'unclassified']
    if len(unclassified):
        issue('G1', 'CONDITION', 'GSE284666', f'{len(unclassified)} samples have an unclassified treatment: {sorted(unclassified.treatment.unique())}')
    if 'GSE284666' in GEO:
        sc = manifest[manifest.study_accession == 'GSE284666']
        print('\nGSE284666 samples per cell_line x condition [GEO]:')
        print(pd.crosstab(sc.cell_line, sc.label_candidate).to_string())
    for acc in ['GSE309666', 'GSE178514']:
        sub = manifest[(manifest.study_accession == acc) & manifest.use_in_mrna_analysis]
        if len(sub):
            key = 'donor' if acc == 'GSE309666' else 'candidate_biological_id'
            print(f'\n{acc} mRNA samples, {key} x passage [GEO]:')
            print(pd.crosstab(sub[key], sub.passage.astype(int)).to_string())

## G1.3 — Supplementary files: inspect, then download an explicit allowlist
Sizes and formats are listed first. Only the three processed tables are fetched, and the size cap stops anything unexpected. The GSE178514 lncRNA and miRNA tables are listed but not fetched, so they cannot be mixed in.

In [ ]:
ALLOWLIST = {'GSE284666_RAW.tar': ('GSE284666', 'tar'),
             'GSE309666_salmon_gene_counts.tsv.gz': ('GSE309666', 'gz'),
             'GSE178514_AllGeneFPKM_mRNA.txt.gz': ('GSE178514', 'gz')}
RAW_LIKE = re.compile(r'\.(fastq|fq|sra|bam|cram)(\.gz)?$', re.I)

inventory = []
for acc in ACCESSIONS:
    urls = GEO.get(acc, {}).get('series', {}).get('Series_supplementary_file', [])
    for u in urls:
        https_url = u.replace('ftp://ftp.ncbi.nlm.nih.gov', 'https://ftp.ncbi.nlm.nih.gov')
        name = https_url.rsplit('/', 1)[-1]
        head = http_head(https_url)
        inventory.append({'accession': acc, 'file': name, 'url': https_url, 'head_status': head['status'], 'declared_bytes': head['bytes'],
                          'last_modified': head['last_modified'], 'raw_read_format': bool(RAW_LIKE.search(name)),
                          'allowlisted': name in ALLOWLIST})
    try:
        listing = http_get(f'{series_dir(acc)}/suppl/', tries=2).text
        listed = sorted(set(re.findall(r'href="([^"/?][^"]*\.(?:gz|tar|txt|tsv|csv|xlsx?|h5|h5ad|bw|zip))"', listing)))
        print(f'{acc} directory listing (best effort, can be empty even when files exist): {listed}')
    except Exception as e:
        print(f'{acc} directory listing unavailable: {e}')

inventory_df = pd.DataFrame(inventory)
for fname, (acc, _) in ALLOWLIST.items():
    if acc in GEO and (inventory_df.empty or fname not in set(inventory_df.file)):
        issue('G1', 'BLOCKER', acc, f'{fname} is not listed in Series_supplementary_file.')

FILES = {}
download_rows = []
for _, row in inventory_df.iterrows() if len(inventory_df) else []:
    if not row.allowlisted:
        continue
    kind = ALLOWLIST[row.file][1]
    if row.declared_bytes and row.declared_bytes > MAX_DOWNLOAD_BYTES:
        issue('G1', 'BLOCKER', row.accession, f'{row.file} is {row.declared_bytes} bytes, above the cap; not downloaded.')
        continue
    rec = fetch_file(row.url, RAW / row.file, kind, int(row.declared_bytes) if pd.notna(row.declared_bytes) else None)
    rec['accession'] = row.accession
    download_rows.append(rec)
    if rec['path']:
        FILES[row.file] = Path(rec['path'])
    else:
        issue('G1', 'BLOCKER', row.accession,
              f"{row.file}: no route gave a valid file. Attempts: {rec['attempts']}. "
              f'Recovery: attach a Kaggle Dataset holding this exact file, from {row.url}')

downloads_df = pd.DataFrame(download_rows)
if len(inventory_df):
    print(inventory_df.drop(columns=['url']).to_string())
if len(downloads_df):
    save_csv(downloads_df.assign(attempts=downloads_df.attempts.map(' ; '.join)), MANIFESTS / 'download_records.csv')
    print(downloads_df[['file', 'route', 'bytes', 'declared_bytes', 'sha256']].to_string())
save_csv(inventory_df, MANIFESTS / 'file_inventory.csv')

## G1.4 — Is each file really a sample-by-gene matrix?
Schema is discovered at runtime and printed, not assumed. Values are stored as deposited. **No normalization, log transform or filtering happens here.** Non-gene summary rows are removed from the per-sample files and counted.

Scale inference is stated as evidence: integer share, column-sum stability, and range. Non-integer tables with unequal column sums can't be distinguished from one another by distribution alone, and the report says so instead of guessing.

In [ ]:
ENS_RE = re.compile(r'^ENSG\d+')
SYMBOL_COL = re.compile(r'^(gene[_. ]?name|gene[_. ]?symbol|symbol|hgnc[_. ]?symbol)$', re.I)
NONGENE_ROW = re.compile(r'^(__|N_unmapped|N_multimapping|N_noFeature|N_ambiguous)')

def norm(s):
    return re.sub(r'[^a-z0-9]', '', str(s).lower())

def read_table_smart(path):
    opener = gzip.open if str(path).endswith('.gz') else open
    with opener(path, 'rt', encoding='utf-8', errors='replace') as f:
        lines = [next(f, '') for _ in range(50)]
    skip = 0
    for line in lines:
        if line.startswith('#') and '\t' not in line:
            skip += 1
        else:
            break
    head = lines[skip] if skip < len(lines) else ''
    sep = '\t' if '\t' in head else ',' if ',' in head else r'\s+'
    engine_opts = {'engine': 'python'} if sep == r'\s+' else {'engine': 'c', 'low_memory': False}
    df = pd.read_csv(path, sep=sep, skiprows=skip, dtype=str, encoding='utf-8', encoding_errors='replace', **engine_opts)
    if not isinstance(df.index, pd.RangeIndex):   # R-style table: header one field shorter than rows
        df = df.reset_index().rename(columns={'index': 'gene_id'})
    df.columns = [str(c).lstrip('#').strip() for c in df.columns]
    return df, {'skipped_comment_lines': skip, 'separator': repr(sep), 'n_rows': len(df), 'columns_first_40': list(df.columns)[:40]}

def map_columns(columns, tokens_by_sample, prefer=None):
    """Exact match, then longest-token containment. Anything still ambiguous stays unmapped: never guessed."""
    tokens = {norm(t): s for s, ts in tokens_by_sample.items() for t in ts if t and len(norm(t)) >= 3}
    hits = {s: [] for s in tokens_by_sample}
    used = set()
    for c in columns:
        n = norm(c)
        if n in tokens:
            hits[tokens[n]].append(c)
            used.add(c)
    for c in columns:
        if c in used:
            continue
        n = norm(c)
        cands = sorted((t for t in tokens if t in n and len(t) >= 4), key=len, reverse=True)
        if cands:
            hits[tokens[cands[0]]].append(c)
    mapping, ambiguous, unmapped = {}, {}, []
    for s, cols in hits.items():
        if prefer and len(cols) > 1:
            cols = [c for c in cols if re.search(prefer, str(c), flags=re.I)] or cols
        if len(cols) == 1:
            mapping[s] = cols[0]
        elif cols:
            ambiguous[s] = cols
        else:
            unmapped.append(s)
    return mapping, ambiguous, unmapped

def annotate(ann):
    """Returns a frame with gene_id, id_type, ensembl, symbol, entrez, aligned to rows."""
    id_col = ann.columns[0]
    ids = ann[id_col].astype(str).str.strip()
    out = pd.DataFrame({'gene_id': ids})
    ens_share = ids.str.match(ENS_RE).mean()
    num_share = ids.str.fullmatch(r'\d+').mean()
    id_type = 'ensembl' if ens_share > 0.5 else 'entrez_like' if num_share > 0.5 else 'symbol_like'
    out['ensembl'] = ids.str.extract(r'^(ENSG\d+)', expand=False)
    out['entrez'] = ids.where(ids.str.fullmatch(r'\d+'))
    sym = next((c for c in ann.columns[1:] if SYMBOL_COL.match(str(c))), None)
    if sym is not None:
        out['symbol'] = ann[sym].astype(str).str.strip().values
    else:
        out['symbol'] = ids.where(~ids.str.match(ENS_RE) & ~ids.str.fullmatch(r'\d+'))
    return out, id_type

def infer_scale(expr):
    v = expr.to_numpy(dtype=float)
    finite = v[np.isfinite(v)]
    if finite.size == 0:
        return {'scale_inferred': 'no numeric values'}
    sums = np.nansum(v, axis=0)
    int_share = float(np.mean(np.isclose(finite, np.round(finite), atol=1e-6)))
    nonzero = finite[finite != 0]
    int_share_nz = float(np.mean(np.isclose(nonzero, np.round(nonzero), atol=1e-6))) if nonzero.size else 1.0
    cv = float(np.std(sums) / np.mean(sums)) if np.mean(sums) else float('nan')
    info = {'integer_share_nonzero': round(int_share_nz, 4), 'min': float(finite.min()), 'max': float(finite.max()),
            'median_col_sum': float(np.median(sums)), 'col_sum_cv': round(cv, 4), 'zero_share': round(float(np.mean(finite == 0)), 4)}
    if finite.min() < 0:
        scale = 'contains negatives: already transformed'
    elif finite.max() < 50 and int_share_nz < 0.9:
        scale = 'small range: possibly already log-scale'
    elif int_share_nz > 0.999:
        scale = 'integer counts'
    elif cv < 0.01 and abs(np.median(sums) - 1e6) / 1e6 < 0.02:
        scale = 'TPM-like (column sums about 1e6)'
    else:
        scale = 'non-integer, unequal column sums: estimated counts or FPKM; not separable from distribution alone'
    info['scale_inferred'] = scale
    return info

def make_dataset(acc, expr, ann, id_type, schema, mapping_info):
    expr = expr.apply(pd.to_numeric, errors='coerce')
    non_numeric = float(expr.isna().mean().mean())
    info = {'accession': acc, 'n_genes': expr.shape[0], 'n_samples': expr.shape[1], 'gene_id_type': id_type,
            'duplicate_gene_ids': int(ann.gene_id.duplicated().sum()), 'na_share': round(non_numeric, 6),
            **infer_scale(expr), 'schema': schema, 'mapping': mapping_info}
    return {'acc': acc, 'expr': expr, 'ann': ann, 'info': info}

DATASETS = {}

def expected_samples(acc):
    return manifest[(manifest.study_accession == acc) & manifest.use_in_mrna_analysis]

def load_matrix(acc, fname, tokens_fn, prefer=None):
    try:
        df, schema = read_table_smart(FILES[fname])
    except Exception as e:
        issue('G1', 'BLOCKER', acc, f'{fname} could not be parsed: {type(e).__name__}: {e}')
        return
    print(f'\n{acc} {fname}: {df.shape[0]} rows, {df.shape[1]} columns')
    print('  columns:', list(df.columns)[:30])
    print(df.iloc[:3, :min(6, df.shape[1])].to_string())
    samples = expected_samples(acc)
    mapping, ambiguous, unmapped = map_columns(df.columns, {r.sample_id: tokens_fn(r) for r in samples.itertuples()}, prefer)
    extra = [c for c in df.columns if c not in mapping.values()]
    print(f'  mapped {len(mapping)}/{len(samples)}; ambiguous {len(ambiguous)}; unmapped {len(unmapped)}; other columns: {extra[:12]}')
    if ambiguous:
        issue('G1', 'CONDITION', acc, f'Columns ambiguous for {len(ambiguous)} samples (not guessed): {dict(list(ambiguous.items())[:3])}')
    if unmapped:
        issue('G1', 'BLOCKER', acc, f'{len(unmapped)} of {len(samples)} samples have no column. Header: {list(df.columns)[:30]}')
    if not mapping:
        return
    ann_cols = [c for c in df.columns if c not in mapping.values()]
    if not ann_cols:
        issue('G1', 'BLOCKER', acc, 'No gene identifier column found.')
        return
    ann, id_type = annotate(df[ann_cols])
    expr = df[list(mapping.values())].rename(columns={v: k for k, v in mapping.items()})
    ds = make_dataset(acc, expr, ann, id_type, schema,
                      {'mapped': len(mapping), 'expected': len(samples), 'ambiguous': len(ambiguous), 'unmapped': len(unmapped),
                       'column_for_sample': mapping, 'annotation_columns': ann_cols})
    DATASETS[acc] = ds

if 'GSE309666' in GEO and 'GSE309666_salmon_gene_counts.tsv.gz' in FILES:
    load_matrix('GSE309666', 'GSE309666_salmon_gene_counts.tsv.gz',
                lambda r: [r.sample_id, r.library_name, f'{r.donor}_p{int(r.passage)}', f'p{int(r.passage)}_{r.donor}', f'donor{r.donor}p{int(r.passage)}', r.title])
if 'GSE178514' in GEO and 'GSE178514_AllGeneFPKM_mRNA.txt.gz' in FILES:
    load_matrix('GSE178514', 'GSE178514_AllGeneFPKM_mRNA.txt.gz',
                lambda r: [r.sample_id, f'P{int(r.passage)}{r.candidate_biological_id}', f'MSC_P{int(r.passage)}_REP{r.replicate}', f'P{int(r.passage)}_REP{r.replicate}'],
                prefer='fpkm')

In [ ]:
# GSE284666_RAW.tar: list members and classify formats BEFORE reading any of them. Nothing is extracted to disk.
MAX_MEMBER_BYTES = 200_000_000
tar_records, sample_series, member_issues, member_schema = [], {}, [], {}

def parse_count_member(raw):
    text = raw.decode('utf-8', 'replace')
    lines = [l for l in text.splitlines() if l.strip() and not (l.startswith('#') and '\t' not in l)]
    if not lines:
        raise ValueError('no rows')
    sep = '\t' if '\t' in lines[0] else ',' if ',' in lines[0] else r'\s+'
    df = pd.read_csv(io.StringIO('\n'.join(lines)), sep=sep, header=None, dtype=str, engine='c' if sep != r'\s+' else 'python')
    has_header = pd.to_numeric(df.iloc[0, -1:], errors='coerce').isna().all()
    header = df.iloc[0].tolist() if has_header else None
    if has_header:
        df = df.iloc[1:]
    ids = df.iloc[:, 0].astype(str).str.strip()
    nongene = ids.str.contains(NONGENE_ROW)
    vals = pd.to_numeric(df.iloc[:, -1], errors='coerce')
    s = pd.Series(vals[~nongene].values, index=ids[~nongene].values)
    return s, {'n_cols': df.shape[1], 'header': header, 'nongene_rows_removed': int(nongene.sum()),
               'duplicate_ids': int(s.index.duplicated().sum()), 'na': int(s.isna().sum())}

if 'GSE284666_RAW.tar' in FILES:
    with tarfile.open(FILES['GSE284666_RAW.tar'], 'r:*') as tf:
        for m in tf.getmembers():
            safe = m.isfile() and not m.name.startswith(('/', '\\')) and '..' not in Path(m.name).parts
            gsm = re.match(r'(GSM\d+)', Path(m.name).name)
            tar_records.append({'member': m.name, 'bytes': m.size, 'is_file': m.isfile(), 'safe_name': safe,
                                'gsm': gsm.group(1) if gsm else '', 'suffixes': ''.join(Path(m.name).suffixes),
                                'raw_read_format': bool(RAW_LIKE.search(m.name))})
        tar_df = pd.DataFrame(tar_records)
        print(f'Tar members: {len(tar_df)}; total {tar_df.bytes.sum():,} compressed bytes')
        print(tar_df.suffixes.value_counts().to_string())
        print('Raw-read formats present:', int(tar_df.raw_read_format.sum()))
        print(tar_df.head(3).to_string())
        if tar_df.raw_read_format.any():
            issue('G1', 'CONDITION', 'GSE284666', f'{int(tar_df.raw_read_format.sum())} tar members look like raw reads; not read.')
        wanted = set(expected_samples('GSE284666').sample_id) if 'GSE284666' in GEO else set()
        for m in tf.getmembers():
            rec = next(r for r in tar_records if r['member'] == m.name)
            if not rec['safe_name'] or rec['raw_read_format'] or rec['gsm'] not in wanted or m.size > MAX_MEMBER_BYTES:
                continue
            try:
                raw = tf.extractfile(m).read()
                if raw[:2] == b'\x1f\x8b':
                    raw = gzip.decompress(raw)
                series_, info = parse_count_member(raw)
                member_schema[rec['gsm']] = info
                if info['duplicate_ids']:
                    series_ = series_[~series_.index.duplicated(keep='first')]
                sample_series[rec['gsm']] = series_
            except Exception as e:
                member_issues.append(f"{m.name}: {type(e).__name__}: {e}")
    save_csv(tar_df, MANIFESTS / 'GSE284666_tar_inventory.csv')
    if member_issues:
        issue('G1', 'CONDITION', 'GSE284666', f'{len(member_issues)} members failed to parse: {member_issues[:3]}')
    if sample_series:
        first_schema = next(iter(member_schema.values()))
        print('First parsed member schema:', first_schema)
        dup_total = sum(v['duplicate_ids'] for v in member_schema.values())
        if dup_total:
            issue('G1', 'CONDITION', 'GSE284666', f'{dup_total} duplicate gene ids across members; first occurrence kept, resolve in notebook 01.')
        expr284 = pd.concat(sample_series, axis=1, join='outer', sort=True)
        gene_sets = {frozenset(s.index) for s in sample_series.values()}
        missing = sorted(wanted - set(expr284.columns))
        print(f'Assembled {expr284.shape[0]} genes x {expr284.shape[1]} samples; distinct gene sets across members: {len(gene_sets)}; samples without a member: {len(missing)}')
        if len(gene_sets) > 1:
            issue('G1', 'CONDITION', 'GSE284666', f'{len(gene_sets)} distinct gene sets across members; absent genes stay NaN, not zero.')
        if missing:
            issue('G1', 'BLOCKER', 'GSE284666', f'{len(missing)} manifest samples have no tar member: {missing[:5]}')
        ann284, id_type = annotate(pd.DataFrame({'gene_id': expr284.index.astype(str)}))
        DATASETS['GSE284666'] = make_dataset('GSE284666', expr284.reset_index(drop=True), ann284, id_type,
                                             {'member_schema_first': first_schema, 'distinct_gene_sets': len(gene_sets)},
                                             {'mapped': expr284.shape[1], 'expected': len(wanted), 'ambiguous': 0, 'unmapped': len(missing)})
        del sample_series

In [ ]:
# Per-sample QC and scale evidence; matrices stored exactly as deposited.
# Claims made by the deposit, compared below with what the file shows.
DEPOSIT_SCALE_CLAIM = {
    'GSE284666': 'GEO: "text files containing counts" (featureCounts)',
    'GSE309666': 'GEO: "gene-level Salmon TPM-like counts"; file name says gene_counts',
    'GSE178514': 'File name says FPKM; GEO processing note says "raw gene counts"',
}
qc_parts = []
for acc, ds in DATASETS.items():
    expr = ds['expr']
    totals, detected = expr.sum(axis=0), (expr > 0).sum(axis=0)
    log_total = np.log10(totals.clip(lower=1))
    def robust_z(x):
        mad = (x - x.median()).abs().median()
        return (x - x.median()) / (1.4826 * mad) if mad > 0 else x * 0
    qc = pd.DataFrame({'sample_id': expr.columns, 'study_accession': acc, 'column_sum': totals.values, 'genes_gt0': detected.values,
                       'zero_share': (expr == 0).mean(axis=0).values, 'na_share': expr.isna().mean(axis=0).values,
                       'z_log_sum': robust_z(log_total).values, 'z_genes': robust_z(detected).values})
    qc['qc_flag'] = np.where((qc.z_log_sum.abs() > 3.5) | (qc.z_genes.abs() > 3.5), 'review', '')
    qc_parts.append(qc)
    out = PROC / f'{acc}_matrix_as_deposited.tsv.gz'
    if not out.exists():
        pd.concat([ds['ann'].reset_index(drop=True), expr.reset_index(drop=True)], axis=1).to_csv(out, sep='\t', index=False, compression='gzip')
    info = {k: v for k, v in ds['info'].items() if k not in ('schema', 'mapping')}
    print(f'\n{acc} [measured]:', json.dumps(info, default=str))
    print('  deposit claim:', DEPOSIT_SCALE_CLAIM[acc])
    flagged = qc[qc.qc_flag != '']
    print(f'  QC review flags: {len(flagged)}', flagged[['sample_id', 'column_sum', 'genes_gt0']].to_dict('records') if len(flagged) else '')
    if len(flagged):
        issue('G1', 'NOTE', acc, f'{len(flagged)} samples flagged for review (robust z > 3.5); none removed.')
    scale = ds['info']['scale_inferred']
    if acc == 'GSE284666' and scale != 'integer counts':
        issue('G1', 'CONDITION', acc, f'Expected integer counts; file shows: {scale}.')
    if acc == 'GSE178514':
        if scale == 'integer counts':
            issue('G1', 'CONDITION', acc, 'Integers despite the FPKM file name: confirm what this table holds before using it.')
        else:
            issue('G1', 'CONDITION', acc, f'{scale}. The GEO "raw gene counts" note is not supported; do not feed this table to DESeq2.')
    if acc == 'GSE309666' and not scale.startswith('TPM-like') and scale != 'integer counts':
        issue('G1', 'CONDITION', acc, f"{scale}. Median column sum {ds['info']['median_col_sum']:,.0f}, so the GEO phrase 'TPM-like' is not supported if this is far from 1e6. Decide its handling in notebook 01.")
    if ds['info']['duplicate_gene_ids']:
        issue('G1', 'NOTE', acc, f"{ds['info']['duplicate_gene_ids']} duplicate gene ids; no aggregation done yet.")
qc_all = pd.concat(qc_parts, ignore_index=True) if qc_parts else pd.DataFrame()
save_csv(qc_all, MANIFESTS / 'sample_qc.csv')
matrix_info = {a: d['info'] for a, d in DATASETS.items()}
versioned_write(MANIFESTS / 'matrix_info.json', json.dumps(matrix_info, indent=1, default=str))

## G1.5 — Metadata reconciliation and published evidence
- **GSE309666:** explains the 18 GEO records against the paper's 17 samples. No sample is discarded.
- **GSE178514:** tests the `generation` field against the passage in the title.
- **Literature:** full text is fetched from PMC, the reference list removed, then quotes are checked and assay terms counted. A statement counts as `[published]` only if verified in this run.

In [ ]:
PAPERS = {
    'GSE309666': {'pmid': '41791726', 'quotes': [
        'The samples represented five donors at passage 1, five at passage 3, two at passage 6, and five at their final passage.',
        'each of the 17 samples',
        'as in its final passage (passage 8)',
        'defined as cessation of population growth where fewer ASC were harvested than seeded',
        'no PDT data for final passage of donor A-C since fewer ASC were harvested than seeded']},
    'GSE178514': {'pmid': '37831180', 'quotes': []},
    'GSE178514_prior_ref27': {'pmid': '34631285', 'quotes': [
        'in MSCs from three donors were used for the subsequent analyses',
        'A total of 20 healthy donors',
        'Percent of SA-β-gal staining in (C)',
        'CCK-8 assay of MSCs at P4, P6, P8, P10, and P12']},
    'GSE284666': {'pmid': '42276073', 'quotes': [
        'The senescent state was confirmed by measuring BrdU incorporation',
        'the top 1000 features were selected using SelectKBest',
        'features with absolute weight >0.05 were extracted',
        'training 14 independent binary classifiers',
        'BMMSC bone marrow mesenchymal cells (ATCC']},
}
TERMS = {'SA_beta_gal': r'beta.?galactosidase|β.?gal|SA.?β', 'p16_CDKN2A': r'\bp16\b|CDKN2A', 'p21_CDKN1A': r'\bp21\b|CDKN1A', 'BrdU_EdU': r'\bBrdU\b|\bEdU\b', 'suffix_393_411_413': r'\b(?:393|411|413)\b'}

def squash(s):
    return re.sub(r'\s+', '', s).lower()

def fetch_paper(pmid):
    summary = http_get(f'{EUTILS}/esummary.fcgi', params={'db': 'pubmed', 'id': pmid, 'retmode': 'json', 'tool': 'SenGuard-MSC'}).json()['result'][pmid]
    ids = {a['idtype']: a['value'] for a in summary.get('articleids', [])}
    rec = {'pmid': pmid, 'title': summary.get('title'), 'journal': summary.get('fulljournalname'), 'pubdate': summary.get('pubdate'),
           'doi': ids.get('doi'), 'pmcid': ids.get('pmc')}
    text = ''
    if rec['pmcid']:
        xml = http_get(f'{EUTILS}/efetch.fcgi', params={'db': 'pmc', 'id': rec['pmcid'].replace('PMC', ''), 'retmode': 'xml', 'tool': 'SenGuard-MSC'}).text
        if '<body' in xml:
            xml = re.sub(r'<ref-list.*?</ref-list>', ' ', xml, flags=re.S)
            text = re.sub(r'\s+', ' ', html.unescape(re.sub(r'<[^>]+>', ' ', xml)))
            rec['text_scope'] = 'PMC full text, references removed'
    if not text:
        text = http_get(f'{EUTILS}/efetch.fcgi', params={'db': 'pubmed', 'id': pmid, 'rettype': 'abstract', 'retmode': 'text', 'tool': 'SenGuard-MSC'}).text
        rec['text_scope'] = 'abstract only: absence of a term here means nothing'
    return rec, text

LIT = {}
for key, spec in PAPERS.items():
    try:
        rec, text = fetch_paper(spec['pmid'])
        rec['chars'] = len(text)
        rec['quote_check'] = {q: squash(q) in squash(text) for q in spec['quotes']}
        rec['term_counts'] = {t: len(re.findall(p, text, flags=re.I)) for t, p in TERMS.items()}
        rec['term_contexts'] = {t: [text[max(0, m.start() - 100):m.end() + 100] for m in list(re.finditer(p, text, flags=re.I))[:4]]
                                for t, p in TERMS.items()}
        LIT[key] = rec
        print(f"{key}: PMID {rec['pmid']} {rec['pmcid']} {rec['journal']} ({rec['pubdate']}) | {rec['text_scope']}\n  {rec['title']}")
        print('  term counts:', rec['term_counts'])
        for q, ok in rec['quote_check'].items():
            print(f"  quote {'VERIFIED' if ok else 'NOT FOUND'}: {q}")
            if not ok:
                issue('G1', 'CONDITION', key, f'Quote not found in fetched text: {q}')
    except Exception as e:
        LIT[key] = {'pmid': spec['pmid'], 'error': f'{type(e).__name__}: {e}'}
        issue('G1', 'CONDITION', key, f'Literature check unavailable: {e}')
versioned_write(MANIFESTS / 'literature_evidence.json', json.dumps(LIT, indent=1, ensure_ascii=False))

def lit_verified(key, quote):
    return LIT.get(key, {}).get('quote_check', {}).get(quote, False)

In [ ]:
# GSE309666: 18 GEO records vs 17 analysed samples
RECON = {}
if 'GSE309666' in GEO:
    asc = manifest[(manifest.study_accession == 'GSE309666')].copy()
    asc['final_passage_for_donor'] = asc.groupby('donor').passage.transform('max') == asc.passage
    asc['matches_paper_design'] = asc.final_passage_for_donor | asc.passage.isin([1, 3]) | ((asc.passage == 6) & asc.donor.isin(['D', 'E']))
    counts = {'p1': int((asc.passage == 1).sum()), 'p3': int((asc.passage == 3).sum()), 'p6': int((asc.passage == 6).sum()),
              'final': int(asc.final_passage_for_donor.sum())}
    extras = asc[~asc.matches_paper_design]
    RECON['GSE309666'] = {'geo_records': len(asc), 'counts_from_geo': counts, 'paper_design': {'p1': 5, 'p3': 5, 'p6': 2, 'final': 5},
                          'records_outside_paper_design': extras[['sample_id', 'donor', 'passage', 'title']].to_dict('records'),
                          'final_passage_by_donor': asc[asc.final_passage_for_donor].set_index('donor').passage.astype(int).to_dict()}
    print('GSE309666 reconciliation:', json.dumps(RECON['GSE309666'], indent=1))
    verified = lit_verified('GSE309666', 'as in its final passage (passage 8)') and lit_verified('GSE309666', 'each of the 17 samples')
    if len(extras) == 1 and counts == {'p1': 5, 'p3': 5, 'p6': 2, 'final': 5}:
        x = extras.iloc[0]
        RECON['GSE309666']['resolution'] = (f"[inference] {x.sample_id} (donor {x.donor}, p{int(x.passage)}) is the 18th record: GEO holds this donor's p{int(x.passage)} and its final passage, "
                                            f"while the paper's design counts only the final one. Paper statement checked in this run: {verified}. The paper does not name this sample, so this is not an author-stated exclusion. Retained and flagged.")
        issue('G1', 'NOTE', 'GSE309666', RECON['GSE309666']['resolution'])
    else:
        RECON['GSE309666']['resolution'] = 'Could not be resolved by the design counts; see records_outside_paper_design.'
        issue('G1', 'CONDITION', 'GSE309666', RECON['GSE309666']['resolution'])

    growth_arrest_verified = lit_verified('GSE309666', 'no PDT data for final passage of donor A-C since fewer ASC were harvested than seeded')
    status = {'A': 'growth_cessation_at_final', 'B': 'growth_cessation_at_final', 'C': 'growth_cessation_at_final',
              'D': 'stopped_at_p10_right_censored', 'E': 'stopped_at_p10_right_censored'}
    idx = manifest.study_accession == 'GSE309666'
    manifest.loc[idx, 'final_passage_for_donor'] = manifest.loc[idx, 'sample_id'].map(asc.set_index('sample_id').final_passage_for_donor)
    manifest.loc[idx, 'matches_paper_design_inferred'] = manifest.loc[idx, 'sample_id'].map(asc.set_index('sample_id').matches_paper_design)
    if growth_arrest_verified:
        manifest.loc[idx, 'published_donor_growth_status'] = manifest.loc[idx, 'donor'].map(status)
        print('Published donor-level growth status is donor-level only: D and E were stopped at passage 10, so their final passage is censored, not "senescent".')

# GSE178514: generation field vs passage; miRNA/mRNA split
if 'GSE178514' in GEO:
    bm = manifest[(manifest.study_accession == 'GSE178514')]
    mrna = bm[bm.use_in_mrna_analysis]
    conflict = int(mrna.generation_conflicts_passage.sum())
    RECON['GSE178514'] = {'gsm_records': len(bm), 'mrna_records': len(mrna), 'mirna_records': int((~bm.use_in_mrna_analysis).sum()),
                          'mrna_generation_conflicts': conflict,
                          'generation_tracks_suffix': bool(mrna.groupby('candidate_biological_id').generation_field.nunique().eq(1).all())}
    print('\nGSE178514 passage vs generation field [GEO]:')
    print(mrna[['sample_id', 'title', 'passage', 'candidate_biological_id', 'generation_field', 'generation_conflicts_passage']].to_string())
    print(RECON['GSE178514'])
    if conflict:
        issue('G1', 'CONDITION', 'GSE178514', f"The generation field conflicts with the title and treatment passage in {conflict}/{len(mrna)} mRNA samples, and tracks the title suffix instead. Passage taken from title and treatment; generation not used. Worth asking the submitters.")
    instrument = sorted(mrna.technology.unique())
    proc_text = ' '.join(GEO['GSE178514']['samples'][0].get('Sample_data_processing', []))
    if 'hiseq' in proc_text.lower() and any('BGISEQ' in t for t in instrument):
        issue('G1', 'NOTE', 'GSE178514', f'Instrument conflict inside the record: {instrument} vs an Illumina HiSeq statement in the processing text.')

## G1.6 — Do GSE178514 "replicates" behave like donors? (exploratory)
GEO doesn't say what suffixes 393, 411 and 413 mean. That decides whether within-subject passage analysis is valid, so it is tested instead of assumed. GSE309666, with stated donors, is the positive control for the method.

Passage effect is removed by centering, then within-group similarity is compared with between-group, using label shuffles within passage. A library-size scale is applied for this test alone.

**Limit:** shared identity can't separate donor from a shared processing batch, and GSE178514 has only a small set of distinct shuffles. Sex-linked genes help only if the groups differ.

In [ ]:
SEX_GENES = {'XIST': ('ENSG00000229807', '7503'), 'RPS4Y1': ('ENSG00000129824', '6192'), 'DDX3Y': ('ENSG00000067048', '8653'),
             'KDM5D': ('ENSG00000012817', '8284'), 'EIF1AY': ('ENSG00000198692', '9086'), 'UTY': ('ENSG00000183878', '7404')}

def to_library_scale(expr):
    return expr.div(expr.sum(axis=0), axis=1) * 1e6

def gene_row(ds, symbol, ens, entrez):
    ann = ds['ann']
    for col, val in (('ensembl', ens), ('symbol', symbol), ('entrez', entrez)):
        hit = np.flatnonzero((ann[col] == val).values)
        if len(hit):
            return hit[0]
    return None

def sex_table(ds, samples):
    scaled = np.log1p(to_library_scale(ds['expr']))
    rows = {}
    for g, (e, z) in SEX_GENES.items():
        i = gene_row(ds, g, e, z)
        rows[g] = scaled.iloc[i] if i is not None else pd.Series(np.nan, index=scaled.columns)
    t = pd.DataFrame(rows).reindex(samples)
    y = t[[g for g in SEX_GENES if g != 'XIST']].mean(axis=1, skipna=True)
    def call(xist, ymean):
        if np.isnan(ymean):
            return 'Y genes absent from table'
        yhi = ymean > 1.5
        if np.isnan(xist):
            return 'Y-high (XIST not in table)' if yhi else 'Y-low (XIST not in table)'
        xhi = xist > 2
        return 'both high' if xhi and yhi else 'XIST-high, Y-low' if xhi else 'Y-high, XIST-low' if yhi else 'both low'
    t['Y_mean'] = y
    t['exploratory_call'] = [call(a, b) for a, b in zip(t.XIST, y)]
    return t.round(2)

def identity_test(ds, sample_df, group_col, bin_col, n_perm=9999, top=2000):
    ids = [s for s in sample_df.sample_id if s in ds['expr'].columns]
    sdf = sample_df.set_index('sample_id').loc[ids]
    x = np.log1p(to_library_scale(ds['expr'][ids]).dropna())
    x = x[(x > np.log1p(1)).mean(axis=1) >= 0.5]
    for b in sdf[bin_col].unique():
        cols = sdf.index[sdf[bin_col] == b]
        x[cols] = x[cols].sub(x[cols].mean(axis=1), axis=0)
    x = x.loc[x.var(axis=1).nlargest(min(top, len(x))).index]
    r = x.rank().corr().to_numpy()
    iu = np.triu_indices(len(ids), 1)
    groups = sdf[group_col].to_numpy()
    bins = sdf[bin_col].to_numpy()
    def stat(g):
        same = g[iu[0]] == g[iu[1]]
        return r[iu][same].mean() - r[iu][~same].mean()
    observed = stat(groups)
    rng = np.random.default_rng(SEED)
    hits = 0
    for _ in range(n_perm):
        g = groups.copy()
        for b in np.unique(bins):
            i = np.flatnonzero(bins == b)
            g[i] = rng.permutation(g[i])
        hits += int(stat(g) >= observed)
    return {'n_samples': len(ids), 'n_genes': len(x), 'within_minus_between_spearman': round(float(observed), 4),
            'perm_p': round(float((1 + hits) / (1 + n_perm)), 4), 'n_perm': n_perm}

IDENTITY = {}
if 'GSE309666' in DATASETS:
    s = manifest[(manifest.study_accession == 'GSE309666') & manifest.use_in_mrna_analysis].copy()
    s['passage_bin'] = np.where(s.passage >= 6, 'p6plus', 'p' + s.passage.astype(int).astype(str))
    IDENTITY['GSE309666_positive_control'] = identity_test(DATASETS['GSE309666'], s, 'donor', 'passage_bin')
    print('GSE309666 stated donors (positive control) [measured]:', IDENTITY['GSE309666_positive_control'])
    t = sex_table(DATASETS['GSE309666'], s.sample_id.tolist())
    print(t.join(s.set_index('sample_id')[['donor', 'passage']]).sort_values(['donor', 'passage']).to_string())
    asc_calls = t.exploratory_call.value_counts().to_dict()
    if len(asc_calls) == 1:
        issue('G1', 'NOTE', 'GSE309666', f'All samples share one sex-linked expression pattern ({asc_calls}). Results may not extend to the other sex; exploratory call, not donor records.')
if 'GSE178514' in DATASETS:
    s = manifest[(manifest.study_accession == 'GSE178514') & manifest.use_in_mrna_analysis].copy()
    s['passage_bin'] = 'p' + s.passage.astype(int).astype(str)
    IDENTITY['GSE178514_suffix'] = identity_test(DATASETS['GSE178514'], s, 'candidate_biological_id', 'passage_bin')
    print('\nGSE178514 title suffix (candidate donor) [measured]:', IDENTITY['GSE178514_suffix'])
    t = sex_table(DATASETS['GSE178514'], s.sample_id.tolist())
    tj = t.join(s.set_index('sample_id')[['candidate_biological_id', 'passage']])
    print(tj.sort_values(['candidate_biological_id', 'passage']).to_string())
    sex_by_suffix = tj.groupby('candidate_biological_id').exploratory_call.agg(lambda c: c.value_counts().to_dict()).to_dict()
    IDENTITY['GSE178514_sex_by_suffix'] = sex_by_suffix
    print('Sex-linked pattern by suffix:', sex_by_suffix)
    pc = IDENTITY.get('GSE309666_positive_control')
    r = IDENTITY['GSE178514_suffix']
    issue('G1', 'CONDITION', 'GSE178514', 'Donor identity is unconfirmed. '
          f"Suffix identity test: effect {r['within_minus_between_spearman']}, p={r['perm_p']}"
          + (f"; positive control effect {pc['within_minus_between_spearman']}, p={pc['perm_p']}" if pc else '')
          + f". Sex-linked pattern by suffix: {sex_by_suffix}. With 3 x 3 samples the smallest reachable p is about 1/36. "
          'A strong result supports shared biological identity but cannot prove donor; only the submitters or papers can confirm.')
if 'GSE284666' in DATASETS:
    s = manifest[(manifest.study_accession == 'GSE284666') & manifest.use_in_mrna_analysis]
    keep = [i for i in s.sample_id if i in DATASETS['GSE284666']['expr'].columns]
    t = sex_table(DATASETS['GSE284666'], keep).join(s.set_index('sample_id')[['cell_line']])
    sex_by_line = t.groupby('cell_line')[['XIST', 'Y_mean']].median().round(2).join(t.groupby('cell_line').exploratory_call.agg(lambda c: c.value_counts().to_dict()))
    print('\nGSE284666 sex-linked expression by cell line, exploratory [measured]. "both high" can indicate a pooled lot:')
    print(sex_by_line.to_string())
    save_csv(sex_by_line.reset_index(), MANIFESTS / 'GSE284666_sex_genes_by_cell_line.csv')
versioned_write(MANIFESTS / 'identity_test.json', json.dumps(IDENTITY, indent=1))

## G1.7 — Gene-marker coverage (feasibility only)
How many SenCat and hUSI features exist in each table. **No score is computed**, so no external result can influence later model choices. Coverage uses identifiers only.

In [ ]:
def find_repo_file(repo, rel):
    p = EXT / repo / rel
    if p.is_file():
        return p
    return find_attached(Path(rel).name)

GENE_INFO_URL = 'https://ftp.ncbi.nlm.nih.gov/gene/DATA/GENE_INFO/Mammalia/Homo_sapiens.gene_info.gz'
ID_BRIDGE = {}
for acc, ds in DATASETS.items():
    if ds['info']['gene_id_type'] != 'entrez_like':
        continue
    rec = fetch_file(GENE_INFO_URL, META / 'Homo_sapiens.gene_info.gz', 'gz')
    if not rec['path']:
        issue('G1', 'CONDITION', acc, f"Entrez ids could not be mapped: gene_info unavailable. Attempts: {rec['attempts']}")
        continue
    gi = pd.read_csv(rec['path'], sep='\t', usecols=['GeneID', 'Symbol', 'dbXrefs'], dtype=str)
    gi['ensembl'] = gi.dbXrefs.str.extract(r'Ensembl:(ENSG\d+)', expand=False)
    gi = gi.set_index('GeneID')
    ann = ds['ann']
    ann['symbol'] = ann.symbol.where(ann.symbol.notna(), ann.entrez.map(gi.Symbol))
    ann['ensembl'] = ann.ensembl.where(ann.ensembl.notna(), ann.entrez.map(gi.ensembl))
    ID_BRIDGE[acc] = {'source': GENE_INFO_URL, 'sha256': rec['sha256'], 'retrieved': ACCESS_DATE, 'n_ids': len(ann),
                      'n_with_symbol': int(ann.symbol.notna().sum()), 'n_with_ensembl': int(ann.ensembl.notna().sum())}
    print(f'{acc} id map:', ID_BRIDGE[acc])
    ann[['gene_id', 'symbol', 'ensembl']].to_csv(PROC / f'{acc}_gene_id_map.tsv', sep='\t', index=False)
    unmapped = len(ann) - ID_BRIDGE[acc]['n_with_symbol']
    if unmapped:
        issue('G1', 'NOTE', acc, f'{unmapped} of {len(ann)} ids have no current symbol in gene_info.')
versioned_write(MANIFESTS / 'gene_id_bridge.json', json.dumps(ID_BRIDGE, indent=1))

ens2sym = {}
if 'GSE309666' in DATASETS:
    a = DATASETS['GSE309666']['ann'].dropna(subset=['ensembl', 'symbol'])
    ens2sym = dict(zip(a.ensembl, a.symbol))

MARKER_FILES = {
    'SenCat_transcriptomics': ('wf-ml-markers-senescence', 'analysis/transcriptomics_loose5000f_tuned_common_features.csv', 'converted_alias', 'gene', 'coef'),
    'hUSI_SenOCLR': ('HUSI', 'sc/SenOCLR_features.csv', None, 'Gene', 'Weight'),
}
coverage_rows, marker_summary = [], {}
for mname, (repo, rel, ens_col, sym_col, w_col) in MARKER_FILES.items():
    path = find_repo_file(repo, rel)
    if path is None:
        issue('G1', 'CONDITION', mname, f'{rel} not found; coverage not computed.')
        continue
    mk = pd.read_csv(path)
    if not {sym_col, w_col} <= set(mk.columns):
        issue('G1', 'CONDITION', mname, f'Unexpected columns {list(mk.columns)[:6]}; coverage not computed.')
        continue
    weights = pd.to_numeric(mk[w_col], errors='coerce').abs()
    marker_summary[mname] = {'file': str(path), 'n_features': len(mk), 'n_nonzero_weight': int((weights > 0).sum()), 'columns': list(mk.columns)}
    for acc, ds in DATASETS.items():
        ann = ds['ann']
        detected = (ds['expr'].fillna(0) > 0).any(axis=1).values
        ens_in, sym_in = set(ann.ensembl.dropna()), set(ann.symbol.dropna())
        ens_det, sym_det = set(ann.ensembl[detected].dropna()), set(ann.symbol[detected].dropna())
        bridged = False
        if not sym_in and ens_in and ens2sym:
            sym_in = {ens2sym[e] for e in ens_in if e in ens2sym}
            sym_det = {ens2sym[e] for e in ens_det if e in ens2sym}
            bridged = True
        if ens_col and ens_in:
            key, universe, det = mk[ens_col].astype(str).str.replace(r'\.\d+$', '', regex=True), ens_in, ens_det
            via = 'ensembl, via NCBI gene_info' if acc in ID_BRIDGE else 'ensembl'
        elif sym_in:
            key, universe, det = mk[sym_col].astype(str), sym_in, sym_det
            via = 'symbol, bridged from the GSE309666 annotation' if bridged else 'symbol, via NCBI gene_info' if acc in ID_BRIDGE else 'symbol'
        else:
            coverage_rows.append({'marker_set': mname, 'accession': acc, 'matched_via': 'no usable identifier', 'gene_id_type': ds['info']['gene_id_type']})
            issue('G1', 'CONDITION', acc, f"{mname}: the table's {ds['info']['gene_id_type']} ids need an id mapping before coverage can be computed.")
            continue
        present = key.isin(universe).values
        coverage_rows.append({'marker_set': mname, 'accession': acc, 'matched_via': via, 'gene_id_type': ds['info']['gene_id_type'],
                              'n_features': len(mk), 'n_present': int(present.sum()), 'pct_present': round(100 * present.mean(), 1),
                              'pct_abs_weight_present': round(100 * weights[present].sum() / weights.sum(), 1),
                              'n_detected_any_sample': int(key.isin(det).sum())})
coverage = pd.DataFrame(coverage_rows)
save_csv(coverage, MANIFESTS / 'marker_coverage.csv')
print(json.dumps(marker_summary, indent=1))
if len(coverage):
    print(coverage.to_string())
if marker_summary.get('SenCat_transcriptomics'):
    m = marker_summary['SenCat_transcriptomics']
    issue('G1', 'CONDITION', 'SenCat_transcriptomics',
          f"The repo marker file has {m['n_features']} features, while the paper's Methods describe a top-1000 selection followed by intersection across 14 models. "
          'Confirm in G2 which feature set is the published score before reproducing it.')

## G1.8 — Feasibility table, gate decision, reports

In [ ]:
def file_status(fname):
    if fname not in FILES:
        return 'not obtained'
    return f'{FILES[fname].stat().st_size:,} bytes, validated'

def matrix_summary(acc):
    d = DATASETS.get(acc)
    if not d:
        return 'matrix not available'
    i = d['info']
    return f"{i['n_genes']:,} genes x {i['n_samples']} samples; ids {i['gene_id_type']}; {i['scale_inferred']}"

def term_note(key):
    rec = LIT.get(key, {})
    if 'term_counts' not in rec:
        return 'literature check unavailable'
    return f"{rec['text_scope']}; term counts {rec['term_counts']}"

def count_samples(acc):
    if not len(manifest):
        return 0
    return int(((manifest.study_accession == acc) & manifest.use_in_mrna_analysis).sum())

def n_unique(acc, col):
    if not len(manifest) or col not in manifest.columns:
        return 0
    return int(manifest.loc[(manifest.study_accession == acc) & manifest.use_in_mrna_analysis, col].replace('', np.nan).nunique())

CHECKED = lambda key, quote, text: (f'[published] {text}' if lit_verified(key, quote) else f'[published, NOT verified in this run] {text}')
sencat_evidence = CHECKED('GSE284666', 'The senescent state was confirmed by measuring BrdU incorporation', 'paradigm-level BrdU and SA-beta-gal confirmation')
asc_evidence = CHECKED('GSE309666', 'no PDT data for final passage of donor A-C since fewer ASC were harvested than seeded',
                       'growth cessation at final passage for donors A-C only; D and E stopped at p10')
feasibility = pd.DataFrame([
    {'study': 'GSE284666 (SenCat)', 'context': f"{n_unique('GSE284666', 'cell_line')} primary cell types incl. BMMSC and preadipocytes; total-RNA NovaSeq; induced senescence",
     'samples': count_samples('GSE284666'),
     'independent_donors': f"not provided; group by cell_line ({n_unique('GSE284666', 'cell_line')})",
     'passages_or_conditions': 'proliferating and vector controls vs senescence paradigms',
     'senescence_evidence': f"[GEO] treatment per sample; {sencat_evidence}. {term_note('GSE284666')}",
     'file': file_status('GSE284666_RAW.tar'), 'matrix': matrix_summary('GSE284666'),
     'supervised_training': 'Yes, as induced condition vs control, with leave-one-cell-type-out',
     'cannot_conclude': 'Donor-independent performance; replicative senescence; potency. Published coefficients were fit with BMMSC in the data, so transport to BM-MSC is not cell-type independent unless BMMSC is held out and refit.'},
    {'study': 'GSE309666 (AD-MSC)', 'context': 'polyA DNBSEQ; human platelet lysate culture',
     'samples': count_samples('GSE309666'),
     'independent_donors': f"{n_unique('GSE309666', 'donor')} stated in GEO",
     'passages_or_conditions': 'p1, p3, p6 (D, E), final p7-p10; donor B has p7 and p8',
     'senescence_evidence': f"{asc_evidence}. {term_note('GSE309666')}",
     'file': file_status('GSE309666_salmon_gene_counts.tsv.gz'), 'matrix': matrix_summary('GSE309666'),
     'supervised_training': 'No. Passage is not a label; use within-donor trajectories',
     'cannot_conclude': 'Per-sample senescent status; assay concordance; potency or safety. Its 40 published biomarkers came from this cohort, so testing them here is circular.'},
    {'study': 'GSE178514 (BM-MSC)', 'context': 'rRNA-depleted total RNA; BGISEQ-500 per GEO; FBS culture',
     'samples': count_samples('GSE178514'),
     'independent_donors': CHECKED('GSE178514_prior_ref27', 'in MSCs from three donors were used for the subsequent analyses', 'three donors') + f"; GEO does not map them to its {n_unique('GSE178514', 'candidate_biological_id')} title suffixes, see identity test",
     'passages_or_conditions': 'P4, P6, P12 x 3',
     'senescence_evidence': CHECKED('GSE178514_prior_ref27', 'Percent of SA-β-gal staining in (C)', 'SA-beta-gal by passage, P4-P12, reported at group level in the prior paper, not per sequenced sample') + f". Prior paper: {term_note('GSE178514_prior_ref27')}",
     'file': file_status('GSE178514_AllGeneFPKM_mRNA.txt.gz'), 'matrix': matrix_summary('GSE178514'),
     'supervised_training': 'No. n=9 with no per-sample assay label; trajectory only',
     'cannot_conclude': 'Donor-level inference unless pairing is confirmed; count-based statistics on an FPKM table; per-sample senescent status.'},
])
save_csv(feasibility, REPORTS / 'G1_feasibility_table.csv')
print(feasibility[['study', 'samples', 'independent_donors', 'file', 'matrix', 'supervised_training']].to_string())

# Sample manifest: final form, keeps every GEO record
manifest_cols = ['sample_id', 'study_accession', 'title', 'source', 'cell_type', 'cell_line', 'donor', 'donor_evidence', 'candidate_biological_id', 'treatment',
                 'passage', 'passage_evidence', 'replicate', 'technology', 'assay', 'library_selection', 'molecule', 'timestamp', 'geo_submission_date',
                 'library_name', 'available_labels', 'label_candidate', 'label_evidence', 'final_passage_for_donor', 'matches_paper_design_inferred',
                 'published_donor_growth_status', 'generation_field', 'generation_conflicts_passage', 'use_in_mrna_analysis', 'sample_supp_files',
                 'source_link', 'evidence_notes']
if len(manifest):
    in_matrix = {s for d in DATASETS.values() for s in d['expr'].columns}
    manifest['in_expression_matrix'] = manifest.sample_id.isin(in_matrix)
    manifest = manifest[[c for c in manifest_cols if c in manifest.columns] + ['in_expression_matrix']]
    save_csv(manifest, ROOT / 'manifests' / 'sample_manifest.csv')
    print('\nManifest rows:', len(manifest), '| in an expression matrix:', int(manifest.in_expression_matrix.sum()))

# Gate decision
for acc in ACCESSIONS:
    if acc not in DATASETS and not any(i['dataset'] == acc and i['severity'] == 'BLOCKER' for i in ISSUES):
        issue('G1', 'BLOCKER', acc, 'No validated expression matrix for this study.')
    d = DATASETS.get(acc)
    if d and d['info']['mapping']['mapped'] != d['info']['mapping']['expected']:
        sev = 'BLOCKER' if d['info']['mapping']['unmapped'] else 'CONDITION'
        if not any(i['dataset'] == acc and i['severity'] == sev for i in ISSUES):
            issue('G1', sev, acc, f"Only {d['info']['mapping']['mapped']}/{d['info']['mapping']['expected']} samples mapped to a column.")
g1_blockers = [i for i in ISSUES if i['severity'] == 'BLOCKER']
g1_conditions = [i for i in ISSUES if i['severity'] == 'CONDITION' and i['gate'] == 'G1']
G1_STATUS = 'BLOCKED' if g1_blockers else 'CONDITIONAL' if g1_conditions else 'PASS'
issues_df = pd.DataFrame(ISSUES)
save_csv(issues_df, REPORTS / 'G1_issues.csv')

all_matrices = all(a in DATASETS for a in ACCESSIONS)
endpoint_text = ('**Proposed, pending your review of this run:** train and reproduce on GSE284666 (induced condition vs control, leave-one-cell-type-out). '
                 'The MSC cohorts carry no assay-confirmed per-sample senescence label in their deposits, so the primary MSC endpoint should be '
                 'within-donor senescence-signature trajectory (GSE309666) with GSE178514 as supporting evidence, reporting no AUROC against passage.'
                 if all_matrices else '**Not proposed:** one or more matrices could not be verified, so no label or endpoint decision can be made from this run.')

recon_lines = '\n'.join(f'- **{k}:** `{json.dumps(v, ensure_ascii=False)}`' for k, v in RECON.items()) or '- Not available.'
report = f"""# G1 data audit: SenGuard-MSC

**Status: {G1_STATUS}** (G0: {G0_STATUS}) · run {RUN_UTC} · notebook {NOTEBOOK_ID} v{NOTEBOOK_VERSION} · seed {SEED} · runtime: {RUNTIME}

Evidence tags: `[measured]` this run · `[GEO]` metadata text · `[published]` literature checked in this run · `[inference]` our reasoning.

## Decision
{endpoint_text}

Label definitions and model architecture are not locked by this document.

## Feasibility by study
{md_table(feasibility)}

## Sources
{md_table(meta_table.drop(columns=['path'])) if len(meta_table) else 'None retrieved.'}

## Files
{md_table(downloads_df[['file', 'route', 'bytes', 'declared_bytes', 'sha256']]) if len(downloads_df) else 'None obtained.'}

## Measured matrix properties
{md_table(pd.DataFrame([{k: v for k, v in d['info'].items() if k not in ('schema', 'mapping')} for d in DATASETS.values()])) if DATASETS else 'None.'}

## Reconciliation
{recon_lines}

## Identity test, exploratory
`{json.dumps(IDENTITY)}`

Shared identity cannot separate donor from batch.

## Marker coverage, identifiers only
{md_table(coverage) if len(coverage) else 'Not computed.'}

## Open issues
{md_table(issues_df[['severity', 'dataset', 'message']]) if len(issues_df) else 'None.'}

## Standing limits
- Senescence is not potency: nothing here supports claims on T-cell suppression, immune function or clinical safety.
- Passage correlates with senescence but is not a binary ground truth.
- Sample count is not donor count.
- No senescence score was computed in this notebook, and no MSC expression value informed any choice in it.
- G2 cautions: `marker_classifier.py` applies log1p itself (avoid double log) and sums coef x expression with no intercept. hUSI min-max scales within a cohort, so report its raw Spearman value when comparing across datasets.
"""
versioned_write(REPORTS / 'G1_DATA_AUDIT.md', report)

## Final output manifest and gate report

In [ ]:
rows = []
for p in sorted(ROOT.rglob('*')):
    if not p.is_file() or 'external' in p.relative_to(ROOT).parts:
        continue
    rows.append({'path': str(p), 'bytes': p.stat().st_size, 'sha256': sha256(p)})
for name in REPOS:
    d = EXT / name
    if d.is_dir():
        rows.append({'path': str(d) + '/', 'bytes': sum(f.stat().st_size for f in d.rglob('*') if f.is_file()), 'sha256': 'directory; see commit in environment_report.json'})
output_manifest = pd.DataFrame(rows)
output_manifest.to_csv(MANIFESTS / 'output_manifest.csv', index=False)

def show_issues(sev):
    subset = [i for i in ISSUES if i['severity'] == sev]
    print(f'\n{sev} ({len(subset)}):')
    for i in subset:
        print(f"  - [{i['dataset']}] {i['message']}")

print('=' * 100)
print(f'G0 STATUS: {G0_STATUS}\nG1 STATUS: {G1_STATUS}\nRuntime: {RUNTIME}   Run: {RUN_UTC}')
print('=' * 100)
print('Inputs:')
print(meta_table[['accession', 'route', 'samples_parsed', 'retrieved']].to_string() if len(meta_table) else '  none')
if len(downloads_df):
    print(downloads_df[['file', 'route', 'bytes', 'url']].to_string())
print('\nMatrices [measured]:')
for a, d in DATASETS.items():
    print(f"  {a}: {d['info']['n_genes']} genes x {d['info']['n_samples']} samples; ids={d['info']['gene_id_type']}; scale={d['info']['scale_inferred']}")
show_issues('BLOCKER')
show_issues('CONDITION')
show_issues('NOTE')
print('\nOutputs:')
print(output_manifest[['path', 'bytes']].to_string())
print('\nNext step: ' + ('Fix the blockers above and rerun; do not start notebook 01.' if G1_STATUS == 'BLOCKED'
      else 'Review reports/G1_DATA_AUDIT.md and fix label definitions before notebook 01.'))
print('Kaggle Save Version required to persist these outputs:', 'yes' if IS_KAGGLE else 'no (local run; files stay in ROOT)')

# SenGuard-MSC · Notebook 01 — Preprocessing and label protocol

Turns the deposited tables from notebook 00 into analysis-ready matrices, and **writes the analysis protocol before any MSC score exists**. External cohorts only get id harmonization and scale conversion here, with no score and no look at passage trends.

In [ ]:
import json, os, subprocess, sys, tempfile, textwrap
from pathlib import Path
import numpy as np, pandas as pd
ROOT_GUESS = (Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()) / 'senguard_msc'
sys.path.insert(0, str(ROOT_GUESS / 'src'))
import senguard as sg
from senguard import ROOT, PROC, RESULTS, REPORTS, MANIFESTS, EXT, DEV, ASC, BM
np.random.seed(sg.SEED)
for d in [RESULTS, sg.FIGS, REPORTS]:
    d.mkdir(parents=True, exist_ok=True)
sg.require(MANIFESTS / 'sample_manifest.csv', PROC / f'{DEV}_matrix_as_deposited.tsv.gz', PROC / f'{ASC}_matrix_as_deposited.tsv.gz', PROC / f'{BM}_matrix_as_deposited.tsv.gz',
           PROC / f'{BM}_gene_id_map.tsv', sg.EXT / 'wf-ml-markers-senescence', sg.EXT / 'HUSI')
print('ROOT', ROOT, '| pandas', pd.__version__, '| numpy', np.__version__)

## Development labels (GSE284666)
The authors' `assemble_anndata.py` defines the rule: treatments `P` and `EV` are controls and every other treatment is senescence-induced. The table below is computed from the GEO treatment text so any disagreement would show.

In [ ]:
manifest = sg.load_manifest()
labels = sg.dev_labels(manifest)
print(pd.crosstab(labels.treatment.str.replace(r'\s*\(.*', '', regex=True), labels.y).to_string())
print('\nSamples by cell type and label:')
print(pd.crosstab(labels.cell_line, labels.y).to_string())
print('\nTotal', len(labels), 'positives', int(labels.y.sum()), 'cell types', labels.cell_line.nunique())
out = labels[['title', 'cell_line', 'cell_type', 'treatment', 'label_candidate', 'y']].rename_axis('sample_id').reset_index()
out.to_csv(PROC / f'{DEV}_labels.csv', index=False)

## Normalization: reproduce the authors' method and verify it
`normalize_counts.py` filters genes and calls PyDESeq2's `fit_size_factors`. Here the same computation is implemented directly so it can be fit on training folds only. It is compared numerically with PyDESeq2 run in a separate environment. If that environment is absent the check is reported as not run.

In [ ]:
counts, ann = sg.load_as_deposited(DEV)
counts = counts.loc[labels.index]
filtered = sg.author_gene_filter(counts)
norm, sf, ref = sg.mor_normalize(filtered)
n_allpos = int((filtered.to_numpy() > 0).all(axis=0).sum())
print(f'{counts.shape[1]} genes -> {filtered.shape[1]} after author filter; genes non-zero in all samples: {n_allpos} (author adds +1 only if below 100)')
print('Size factors: min %.3f median %.3f max %.3f' % (sf.min(), sf.median(), sf.max()))

venv_py = ROOT / '.venv_check/Scripts/python.exe'
if not venv_py.exists():
    venv_py = ROOT / '.venv_check/bin/python'
check = {'status': 'not run: PyDESeq2 environment absent'}
if venv_py.exists():
    with tempfile.TemporaryDirectory() as td:
        td = Path(td)
        filtered.to_csv(td / 'c.csv')
        code = textwrap.dedent('''
            import sys, json, pandas as pd, anndata as ad, pydeseq2
            from pydeseq2.dds import DeseqDataSet
            c = pd.read_csv(sys.argv[1], index_col=0).astype(int)
            a = ad.AnnData(X=c.values, obs=pd.DataFrame({'g': ['a'] * len(c)}, index=c.index.astype(str)), var=pd.DataFrame(index=c.columns.astype(str)))
            d = DeseqDataSet(adata=a, design='~1', quiet=True)
            d.fit_size_factors()
            pd.Series(d.obs['size_factors'].values if 'size_factors' in d.obs else d.obsm['size_factors'], index=c.index).to_csv(sys.argv[2])
            print(pydeseq2.__version__)
        ''')
        (td / 'run.py').write_text(code)
        clean_env = {k: v for k, v in os.environ.items() if k != 'MPLBACKEND'}   # the notebook kernel's inline backend breaks matplotlib in the other environment
        p = subprocess.run([str(venv_py), '-I', str(td / 'run.py'), str(td / 'c.csv'), str(td / 'sf.csv')], capture_output=True, text=True, timeout=900, env=clean_env)
        if p.returncode == 0:
            sf_ref = pd.read_csv(td / 'sf.csv', index_col=0).iloc[:, 0]
            diff = np.abs(sf_ref.loc[sf.index] / sf - 1)
            check = {'status': 'ran', 'pydeseq2': p.stdout.strip().splitlines()[-1], 'max_relative_difference': float(diff.max()), 'median_relative_difference': float(diff.median())}
        else:
            check = {'status': 'failed', 'stderr': p.stderr[-800:]}
print('Check against PyDESeq2:', json.dumps(check))

norm.T.rename_axis('gene_id').to_csv(PROC / f'{DEV}_norm_counts.tsv.gz', sep='\t')
pd.DataFrame({'size_factor': sf}).rename_axis('sample_id').to_csv(PROC / f'{DEV}_size_factors.csv')

## External cohorts: scale handling and id harmonization
- **GSE309666:** the table is Salmon-derived with Ensembl v113 ids. Median-of-ratios is fit within the cohort without using passage.
- **GSE178514:** FPKM only, so median-of-ratios does not apply. FPKM is rescaled exactly to TPM within the table. Its denominator is the mRNA table, a limit that is carried forward.
- **Ids:** GSE178514 Entrez ids map to Ensembl through NCBI gene_info, with a fallback through symbols in the GSE309666 annotation. This uses annotation only, not expression.

In [ ]:
asc, asc_ann = sg.load_as_deposited(ASC)
asc_ann = asc_ann.copy()
asc_filtered = sg.author_gene_filter(asc)
asc_norm, asc_sf, asc_ref = sg.mor_normalize(asc_filtered)
print(f'{ASC}: {asc.shape[1]} genes -> {asc_filtered.shape[1]} after author filter; size factors {asc_sf.min():.3f}-{asc_sf.max():.3f}; genes non-zero in all samples {len(asc_ref)}')
asc_norm.T.rename_axis('gene_id').to_csv(PROC / f'{ASC}_norm_counts.tsv.gz', sep='\t')

bm, bm_ann = sg.load_as_deposited(BM)
idmap = pd.read_csv(PROC / f'{BM}_gene_id_map.tsv', sep='\t', dtype=str).set_index('gene_id')
sym2ens = asc_ann.dropna(subset=['symbol']).drop_duplicates('symbol', keep=False).set_index('symbol').ensembl
idmap['ensembl_final'] = idmap.ensembl.where(idmap.ensembl.notna(), idmap.symbol.map(sym2ens))
bm_tpm = sg.tpm_from_fpkm(bm)
with_ens = idmap.ensembl_final.reindex(bm_tpm.columns)
print(f'{BM}: {bm.shape[1]} ids; Ensembl via gene_info {int(idmap.ensembl.notna().sum())}; after symbol fallback {int(idmap.ensembl_final.notna().sum())}; duplicated Ensembl {int(with_ens.dropna().duplicated().sum())}')
bm_ens = bm_tpm.loc[:, with_ens.notna()].T.groupby(with_ens.dropna().values).sum().T
print(f'  matrix by Ensembl: {bm_ens.shape}. Duplicates are summed, since transcripts of one gene add on the TPM scale.')
bm_ens.T.rename_axis('gene_id').to_csv(PROC / f'{BM}_tpm_ensembl.tsv.gz', sep='\t')
bm_sym = bm_tpm.loc[:, idmap.symbol.reindex(bm_tpm.columns).notna()]
idmap.reset_index().to_csv(PROC / f'{BM}_gene_id_map_final.tsv', sep='\t', index=False)

common = {
    'dev_filtered_genes': filtered.shape[1],
    'asc_filtered_genes': asc_filtered.shape[1],
    'bm_ensembl_genes': bm_ens.shape[1],
    'dev_and_asc': len(set(filtered.columns) & set(asc_filtered.columns)),
    'dev_and_bm': len(set(filtered.columns) & set(bm_ens.columns)),
    'all_three': len(set(filtered.columns) & set(asc_filtered.columns) & set(bm_ens.columns)),
}
print(common)
json.dump({'pydeseq2_check': check, 'gene_overlap': common}, open(MANIFESTS / 'preprocess_report.json', 'w'), indent=1)

## Analysis protocol, written before any MSC score
This is written to a file and hashed so later notebooks can show it wasn't changed after seeing external results.

In [ ]:
protocol = """# SenGuard-MSC analysis protocol

Fixed on 2026-11-01, before any senescence score was computed on an MSC cohort.
It follows G1 and the endpoint the project lead approved.

## Endpoints
1. Development (GSE284666): induced senescence vs control, 14 cell types. Unit of held-out validation: cell type.
2. Primary MSC endpoint (GSE309666): within-donor change in the published SenCat score across passage.
3. Supporting MSC evidence (GSE178514): same statistic in three putative donors, FPKM-derived.
4. Labelled MSC check: the BMMSC cell type held out of development folds. This is one cell line, so it is a case, not a population estimate.

Passage is not a label. No AUROC, accuracy or calibration is computed against passage.

## Model selection rule (development data only)
- Metric: mean over held-out cell types of the within-cell-type AUROC from nested leave-one-cell-type-out.
- All fold-dependent steps (reference, gene filters, feature selection, tuning, threshold) use training cell types only.
- Choice: highest mean. Any candidate within 0.01 of the best counts as tied, and the simplest wins, in the order
  logreg_l1, elastic_net, linear_svm, rbf_svm, random_forest, xgboost.
- Weights and models are saved with hashes before notebook 03 loads MSC scores.

## External statistics
- Per donor: slope of score per passage (primary), Spearman rho, last minus first passage.
- Across donors: mean slope with an exact two-sided sign-flip test. Five donors give a smallest possible p of 0.0625 and three give 0.25,
  so no conventional significance is reachable by design; effect sizes and direction are the result.
- Competitive null: 1,000 draws shuffling weights among the same genes, and 1,000 draws of random expressed genes. Two-sided empirical p.
- Primary score: published SenCat weights. All other scores (hUSI, SenMayo, Hallmark proliferation, refit models, panels) are secondary and reported without selection.
- Independence: refit weights that exclude BMMSC, and exclude BMMSC plus preadipocytes, are the cell-type-independent sensitivity analysis.

## Handling
- External samples are never removed on score grounds; GSM9271440 (donor B p7) stays in, with a sensitivity analysis excluding it.
- Thresholds are never applied to external cohorts, because no labels exist to justify them. Model outputs there are decision values, not probabilities.
- Null and negative results get reported the same way as positive ones.
"""
versioned = REPORTS / 'ANALYSIS_PROTOCOL.md'
if versioned.exists() and versioned.read_text(encoding='utf-8') != protocol:
    raise RuntimeError('ANALYSIS_PROTOCOL.md already exists with different text. Protocol changes must be made and recorded by hand, not overwritten on rerun.')
versioned.write_text(protocol, encoding='utf-8')
print('Protocol sha256', sg.sha256(versioned))

In [ ]:
outs = [PROC / f'{DEV}_labels.csv', PROC / f'{DEV}_norm_counts.tsv.gz', PROC / f'{DEV}_size_factors.csv', PROC / f'{ASC}_norm_counts.tsv.gz', PROC / f'{BM}_tpm_ensembl.tsv.gz',
        PROC / f'{BM}_gene_id_map_final.tsv', MANIFESTS / 'preprocess_report.json', versioned]
print(sg.output_manifest(outs).assign(path=lambda d: d.path.str.replace(str(ROOT), '', regex=False))[['path', 'bytes']].to_string())

# SenGuard-MSC · Notebook 02 — Reproduce baselines and benchmark on development data

G2 on GSE284666 only. **No MSC cohort is read in this notebook except the BMMSC cell type, which is part of SenCat itself.** It ends by freezing weights and a model choice under the rule in `ANALYSIS_PROTOCOL.md`.

Sections: confounding structure, published-score reproduction, feature-selection leakage, hUSI and pathway scores, nested leave-one-cell-type-out benchmark, model freeze.

In [ ]:
import json, os, sys, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
ROOT_GUESS = (Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()) / 'senguard_msc'
sys.path.insert(0, str(ROOT_GUESS / 'src'))
import senguard as sg
from senguard import ROOT, PROC, RESULTS, REPORTS, MANIFESTS, CKPT, DEV
import sklearn, xgboost
import matplotlib.pyplot as plt
sg.set_fig_style()
np.random.seed(sg.SEED)
CKPT.mkdir(exist_ok=True)
sg.require(PROC / f'{DEV}_labels.csv', PROC / f'{DEV}_norm_counts.tsv.gz', REPORTS / 'ANALYSIS_PROTOCOL.md')
print('sklearn', sklearn.__version__, '| xgboost', xgboost.__version__, '| seed', sg.SEED, '| protocol', sg.sha256(REPORTS / 'ANALYSIS_PROTOCOL.md')[:16])

labels = pd.read_csv(PROC / f'{DEV}_labels.csv').set_index('sample_id')
counts_all, ann = sg.load_as_deposited(DEV)
counts_all = counts_all.loc[labels.index]
norm = pd.read_csv(PROC / f'{DEV}_norm_counts.tsv.gz', sep='\t', index_col=0).T.loc[labels.index]
y, groups = labels.y.to_numpy(), labels.cell_line.to_numpy()
print('counts', counts_all.shape, '| norm', norm.shape, '| positives', int(y.sum()), '| cell types', len(np.unique(groups)))

## Confounding structure [measured]
How much expression variance follows cell type versus senescence label, from a PCA on log-normalized genes. PCA here is descriptive and fits nothing used later.

In [ ]:
from sklearn.decomposition import PCA
xlog = np.log1p(norm)
top = xlog.var().nlargest(5000).index
Z = xlog[top] - xlog[top].mean()
pca = PCA(n_components=10, random_state=sg.SEED).fit(Z)
pcs = pca.transform(Z)
def r2(pc, factor):
    d = pd.DataFrame({'pc': pc, 'f': factor})
    return 1 - d.groupby('f').pc.transform(lambda v: v - v.mean()).pow(2).sum() / (d.pc - d.pc.mean()).pow(2).sum()
paradigm = labels.treatment.str.replace(r'\s*\(.*', '', regex=True).to_numpy()
conf = pd.DataFrame({'PC': np.arange(1, 11), 'var_explained': pca.explained_variance_ratio_, 'r2_cell_type': [r2(pcs[:, i], groups) for i in range(10)],
                     'r2_senescence_label': [r2(pcs[:, i], y) for i in range(10)], 'r2_treatment': [r2(pcs[:, i], paradigm) for i in range(10)]}).round(3)
conf.to_csv(RESULTS / 'dev_confounding_pca.csv', index=False)
print(conf.to_string(index=False))
print('\nLabel is balanced within cell types by design, but treatment dose and type differ by cell type (see manifest), and no donor information exists.')

## Reproducing the published SenCat score
1. **Coefficient reproduction:** refit `ml_classifier.py` at config `loose5000f` on GEO counts and compare with the repo's coefficients. The authors' input was a supplementary table not in GEO, so an exact match isn't guaranteed.
2. **Score:** published weights on median-of-ratios counts, log1p applied once. These weights were fit on these same samples, so this measures self-consistency and is **not** validation.

In [ ]:
pub = sg.load_sencat_markers()
pub_coef = pub.coef
t0 = time.time()
re_coef, re_folds = sg.author_pipeline(norm, y, groups)
print(f'Refit in {time.time() - t0:.1f}s: {len(re_coef)} features')
shared = re_coef.index.intersection(pub_coef.index)
from scipy import stats
repro = {'published_features': len(pub_coef), 'refit_features': len(re_coef), 'overlap': len(shared), 'overlap_share_of_published': len(shared) / len(pub_coef),
         'spearman_coef': float(stats.spearmanr(pub_coef.loc[shared], re_coef.loc[shared]).statistic), 'pearson_coef': float(np.corrcoef(pub_coef.loc[shared], re_coef.loc[shared])[0, 1]),
         'sign_agreement': float((np.sign(pub_coef.loc[shared]) == np.sign(re_coef.loc[shared])).mean())}
print(json.dumps(repro, indent=1))

pub_score, n_used = sg.weighted_log1p_score(norm, pub_coef)
refit_score, _ = sg.weighted_log1p_score(norm, re_coef)
print(f'Published weights: {n_used}/{len(pub_coef)} features in the filtered matrix. Correlation published vs refit score: {np.corrcoef(pub_score, refit_score)[0, 1]:.4f}')
pub_auc = sg.score_by_group_auc(pub_score.to_numpy(), y, groups)
print(pub_auc.round(3).to_string(index=False))
print('Published score, mean within-cell-type AUROC (self-consistency, not validation): %.3f' % pub_auc.auroc.mean())

## Measuring feature-selection leakage in the authors' design
`ml_classifier.py` runs `SelectKBest` on all samples before leave-one-cell-type-out. The comparison below uses the same logistic model and the same `k`:
- **as published:** selection sees the held-out cell type,
- **clean:** selection, gene filters and the normalization reference use training cell types only.

The score in each fold comes from that fold's model, so it is out-of-sample for the fit in both arms.

In [ ]:
def author_loco_scores(counts, y, groups, clean, k=5000):
    rows = []
    if not clean:
        n_all = sg.mor_normalize(sg.author_gene_filter(counts))[0]
        pre = np.log1p(sg.author_feature_prefilter(n_all))
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            cols = pre.columns[sg.SelectKBest(sg.f_classif, k=min(k, pre.shape[1])).fit(pre.to_numpy(), y).get_support(indices=True)]
    for g in np.unique(groups):
        tr = groups != g
        if clean:
            genes, ref = sg.prefilter_train(counts.loc[tr])
            Xtr, Xte = sg.fold_transform(counts.loc[tr], genes, ref), sg.fold_transform(counts.loc[~tr], genes, ref)
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                use = Xtr.columns[sg.SelectKBest(sg.f_classif, k=min(k, Xtr.shape[1])).fit(Xtr.to_numpy(), y[tr]).get_support(indices=True)]
            Xtr, Xte = Xtr[use], Xte[use]
        else:
            Xtr, Xte = pre.loc[tr, cols], pre.loc[~tr, cols]
        m = sg.LogisticRegression(solver='liblinear').fit(Xtr.to_numpy(), y[tr])
        s = m.decision_function(Xte.to_numpy())
        rows.append({'group': g, 'auroc': sg.roc_auc_score(y[~tr], s), 'auprc': sg.average_precision_score(y[~tr], s), 'n': int((~tr).sum())})
    return pd.DataFrame(rows)

leak = author_loco_scores(counts_all, y, groups, clean=False).merge(author_loco_scores(counts_all, y, groups, clean=True), on=['group', 'n'], suffixes=('_as_published', '_clean'))
leak['difference'] = leak.auroc_as_published - leak.auroc_clean
leak.to_csv(RESULTS / 'dev_feature_selection_leakage.csv', index=False)
print(leak.round(3).to_string(index=False))
print('Mean AUROC as published %.4f | clean %.4f | mean difference %.4f' % (leak.auroc_as_published.mean(), leak.auroc_clean.mean(), leak.difference.mean()))

## hUSI and simple pathway scores
- **hUSI:** per-sample Spearman with published weights. Its AUROC is unaffected by the authors' min-max step within a cell type only when computed on raw Spearman, so raw values are used. These weights were trained on other public data, so SenCat is an out-of-training test for this score, assuming no sample overlap, which `trainSet_meta843.csv` is checked for below.
- **Pathway scores:** rank-based and fitted to nothing: SenMayo, Hallmark E2F and G2M (expected to fall), canonical five-marker transcripts from the authors' config.

In [ ]:
h, extra = sg.load_gene_sets()
sym_ann = ann.reindex(norm.columns)
ens2sym = pd.read_csv(PROC / 'GSE309666_matrix_as_deposited.tsv.gz', sep='\t', usecols=['ensembl', 'symbol'], dtype=str).dropna().drop_duplicates('ensembl').set_index('ensembl').symbol
ann_sym = pd.DataFrame({'symbol': ens2sym.reindex(norm.columns)}, index=norm.columns)
print('Ensembl-to-symbol map is annotation only (GSE309666 file columns gene_id and gene_name); genes mapped:', int(ann_sym.symbol.notna().sum()), 'of', norm.shape[1])
norm_sym = sg.to_symbols(norm, ann_sym)
husi_w = sg.load_husi_weights()
husi, n_h = sg.husi_raw(norm_sym, husi_w)
print(f'hUSI features used: {n_h}/{len(husi_w)}')
meta = pd.read_csv(sg.EXT / 'HUSI/Data/trainSet_meta843.csv', low_memory=False, encoding='latin-1')  # file is not UTF-8
txt = meta.astype(str).agg(' '.join, axis=1)
print('hUSI training metadata rows', len(meta), '| columns', list(meta.columns)[:8])
print('  Mentions of the three cohort accessions:', {a: int(txt.str.contains(a).sum()) for a in ['GSE284666', 'GSE309666', 'GSE178514']})
print('  Mentions of MSC or mesenchymal:', int(txt.str.contains(r'mesenchym|\bMSC', case=False, regex=True).sum()))

scores = {'SenCat published (self-consistency)': pub_score, 'hUSI raw': husi}
n_sets = {}
for name, genes, sign in [('SenMayo', extra['SenMayo'], 1), ('Hallmark E2F (inverted)', h['HALLMARK_E2F_TARGETS'], -1), ('Hallmark G2M (inverted)', h['HALLMARK_G2M_CHECKPOINT'], -1)]:
    s, n = sg.rank_set_score(norm_sym, genes)
    scores[name] = sign * s
    n_sets[name] = n
canon = pd.read_csv(sg.EXT / 'wf-ml-markers-senescence/config/Canonical_gene_markers.csv', index_col=0).gene
print('Gene-set coverage', n_sets)
can_scores = {}
for ens, g in canon.items():
    if ens in norm.columns:
        s = np.log1p(norm[ens])
        can_scores[f'{g}{" (inverted)" if g == "LMNB1" else ""}'] = -s if g == 'LMNB1' else s
scores.update(can_scores)
simple_rows, per_type = [], {}
for name, s in scores.items():
    a = sg.score_by_group_auc(s.to_numpy(), y, groups)
    per_type[name] = a.set_index('group').auroc
    m, lo, hi = sg.bootstrap_mean_ci(a.auroc)
    simple_rows.append({'score': name, 'mean_within_type_auroc': m, 'ci_low': lo, 'ci_high': hi, 'min_type_auroc': a.auroc.min(), 'pooled_auroc': sg.roc_auc_score(y, s), 'BMMSC': per_type[name]['BMMSC'], 'prAD': per_type[name]['prAD']})
simple = pd.DataFrame(simple_rows).sort_values('mean_within_type_auroc', ascending=False)
simple.to_csv(RESULTS / 'dev_simple_scores.csv', index=False)
print(simple.round(3).to_string(index=False))
pd.DataFrame(scores).rename_axis('sample_id').to_csv(RESULTS / 'dev_simple_scores_per_sample.csv')

## Nested leave-one-cell-type-out benchmark [measured]
All 14 cell types take turns as the test set. In each outer fold the reference, gene filters, selection, hyperparameters and threshold come from training cell types, with inner leave-one-cell-type-out tuning.

XGBoost uses the GPU when one works, at the project lead's request; its speed and CPU/GPU agreement on a real fold are printed so the choice stays auditable. scikit-learn models have no GPU path and run in parallel on CPU; XGBoost folds run one at a time to stay within VRAM. Results are checkpointed, so a rerun loads them instead of refitting.

## Optional: cuML versus scikit-learn on one real fold
Runs only if RAPIDS is present. Speed and agreement are reported; the benchmark does not switch libraries on this basis.

In [ ]:
# Optional GPU comparison requested for classical ML. It only reports: the benchmark above and below keeps scikit-learn,
# because cuML results may not match numerically and the protocol requires documented equivalence before substituting.
import time as _t
rows = []
try:
    import cuml
    from cuml.linear_model import LogisticRegression as CLR
    from cuml.svm import SVC as CSVC
    from cuml.ensemble import RandomForestClassifier as CRF
    from sklearn.preprocessing import StandardScaler as _SS
    held = 'BMMSC'
    tr = groups != held
    genes, ref_ = sg.prefilter_train(counts_all.loc[tr])
    Xa, Xb = sg.fold_transform(counts_all.loc[tr], genes, ref_), sg.fold_transform(counts_all.loc[~tr], genes, ref_)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        sel = sg.SelectKBest(sg.f_classif, k=2000).fit(Xa.to_numpy(), y[tr])
    Xa, Xb = sel.transform(Xa.to_numpy()), sel.transform(Xb.to_numpy())
    ss = _SS().fit(Xa)
    Xa, Xb = ss.transform(Xa).astype('float32'), ss.transform(Xb).astype('float32')
    ya = y[tr].astype('float32')
    pairs = [('logistic L2', sg.LogisticRegression(C=0.1, solver='lbfgs', max_iter=2000), CLR(C=0.1, max_iter=2000)),
             ('rbf svm', sg.SVC(C=1.0, gamma='scale', probability=True, random_state=sg.SEED), CSVC(C=1.0, gamma='scale', probability=True)),
             ('random forest', sg.RandomForestClassifier(n_estimators=500, max_features='sqrt', random_state=sg.SEED, n_jobs=-1), CRF(n_estimators=500, max_features='sqrt', random_state=sg.SEED))]
    for name, sk, cm in pairs:
        t0 = _t.time(); sk.fit(Xa, y[tr]); t_sk = _t.time() - t0
        ps = sk.predict_proba(Xb)[:, 1]
        t0 = _t.time(); cm.fit(Xa, ya); t_cm = _t.time() - t0
        pc = cm.predict_proba(Xb)
        pc = pc.to_numpy() if hasattr(pc, 'to_numpy') else (pc.get() if hasattr(pc, 'get') else np.asarray(pc))
        pc = pc[:, 1] if pc.ndim == 2 else pc
        rows.append({'model': name, 'sklearn_fit_s': round(t_sk, 2), 'cuml_fit_s': round(t_cm, 2), 'sklearn_auroc': sg.roc_auc_score(y[~tr], ps), 'cuml_auroc': sg.roc_auc_score(y[~tr], pc),
                     'spearman_scores': float(stats.spearmanr(ps, pc).statistic), 'cuml_version': cuml.__version__})
    print(pd.DataFrame(rows).round(3).to_string(index=False))
except Exception as e:
    rows.append({'status': f'cuML comparison not available: {type(e).__name__}: {str(e)[:300]}'})
    print(rows[-1])
pd.DataFrame(rows).to_csv(RESULTS / 'cuml_equivalence.csv', index=False)

In [ ]:
ckpt = CKPT / 'nested_loco_results.json'
held = 'BMMSC'
tr = groups != held
g_genes, g_ref = sg.prefilter_train(counts_all.loc[tr])
Xs = sg.fold_transform(counts_all.loc[tr], g_genes, g_ref)
Xs = Xs[Xs.var().nlargest(2000).index].to_numpy()
Xs_te = sg.fold_transform(counts_all.loc[~tr], g_genes, g_ref)[sg.fold_transform(counts_all.loc[tr], g_genes, g_ref).var().nlargest(2000).index].to_numpy()
speed, xgb_preds = {}, {}
for dev in ['cpu', 'cuda']:
    try:
        t0 = time.time()
        m = xgboost.XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.5, tree_method='hist', device=dev, n_jobs=4, random_state=sg.SEED, eval_metric='logloss').fit(Xs, y[tr])
        speed[dev] = round(time.time() - t0, 2)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            xgb_preds[dev] = m.predict_proba(Xs_te)[:, 1]
    except Exception as e:
        speed[dev] = f'failed: {type(e).__name__}: {e}'
print('XGBoost fit seconds on one real fold:', speed)
xgb_device = 'cuda' if isinstance(speed.get('cuda'), float) else 'cpu'
if len(xgb_preds) == 2:
    print('CPU vs GPU on the held-out fold: max abs probability difference %.5f; AUROC %.4f vs %.4f' % (np.abs(xgb_preds['cpu'] - xgb_preds['cuda']).max(), sg.roc_auc_score(y[~tr], xgb_preds['cpu']), sg.roc_auc_score(y[~tr], xgb_preds['cuda'])))
print('Device used for XGBoost:', xgb_device)

if ckpt.exists():
    results = json.loads(ckpt.read_text())
    print('Loaded checkpoint', ckpt.name, len(results), 'fold results')
else:
    t0 = time.time()
    grid = sg.model_grid(xgb_device)
    cpu_ckpt = CKPT / 'nested_loco_cpu_models.json'
    if cpu_ckpt.exists():
        cpu_results = json.loads(cpu_ckpt.read_text())
    else:
        cpu_results = sg.run_nested_loco(counts_all, y, groups, {k: v for k, v in grid.items() if k != 'xgboost'}, n_jobs=max(1, os.cpu_count() or 2))
        cpu_ckpt.write_text(json.dumps(cpu_results))
    print(f'scikit-learn models on CPU: {(time.time() - t0) / 60:.1f} min')
    t1 = time.time()
    xgb_results = sg.run_nested_loco(counts_all, y, groups, {'xgboost': grid['xgboost']}, n_jobs=1 if xgb_device == 'cuda' else max(1, os.cpu_count() or 2))
    print(f'XGBoost on {xgb_device}: {(time.time() - t1) / 60:.1f} min')
    results = cpu_results + xgb_results
    ckpt.write_text(json.dumps(results))
    print(f'Fit {len(results)} outer folds in {(time.time() - t0) / 60:.1f} min')

In [ ]:
rows, preds = [], []
for r in results:
    m = sg.fold_metrics(r['y'], r['score'], r['threshold'])
    rows.append({'model': r['model'], 'held_out': r['held_out'], **m, 'threshold': r['threshold'], 'inner_auc': r['inner_auc'], 'best_params': json.dumps(r['best_params'])})
    preds += [{'model': r['model'], 'held_out': r['held_out'], 'sample_id': s, 'y': a, 'score': b, 'threshold_from_training': r['threshold'], 'call': int(b >= r['threshold'])} for s, a, b in zip(r['sample_id'], r['y'], r['score'])]
fold_df = pd.DataFrame(rows)
pred_df = pd.DataFrame(preds)
fold_df.to_csv(RESULTS / 'dev_nested_loco_folds.csv', index=False)
pred_df.to_csv(RESULTS / 'dev_nested_loco_predictions.csv', index=False)
pd.DataFrame({'sample_id': labels.index, 'fold_held_out_cell_type': groups, 'y': y}).to_csv(RESULTS / 'dev_fold_assignments.csv', index=False)

ORDER = ['logreg_l1', 'elastic_net', 'linear_svm', 'rbf_svm', 'random_forest', 'xgboost']
summ = []
for name in ORDER:
    f = fold_df[fold_df.model == name]
    m, lo, hi = sg.bootstrap_mean_ci(f.auroc)
    bm, bl, bh = sg.bootstrap_mean_ci(f.balanced_acc)
    summ.append({'model': name, 'mean_auroc': m, 'ci_low': lo, 'ci_high': hi, 'min_auroc': f.auroc.min(), 'worst_type': f.loc[f.auroc.idxmin(), 'held_out'], 'mean_auprc': f.auprc.mean(), 'balanced_acc': bm, 'ba_ci_low': bl, 'ba_ci_high': bh,
                 'macro_f1': f.macro_f1.mean(), 'BMMSC_auroc': f[f.held_out == 'BMMSC'].auroc.iloc[0], 'prAD_auroc': f[f.held_out == 'prAD'].auroc.iloc[0]})
summary = pd.DataFrame(summ)
summary.to_csv(RESULTS / 'dev_model_summary.csv', index=False)
print(summary.round(3).to_string(index=False))
print('\nAUROC by held-out cell type:')
wide = fold_df.pivot(index='held_out', columns='model', values='auroc')[ORDER]
wide.to_csv(RESULTS / 'dev_auroc_by_cell_type.csv')
print(wide.round(3).to_string())
print('\nChosen hyperparameters per model (count over 14 folds):')
for name in ORDER:
    print(' ', name, fold_df[fold_df.model == name].best_params.value_counts().to_dict())

## Which conditions are hard? [measured]
Near-perfect AUROC within most cell types raises the question of what is actually being separated. This breaks held-out errors down by condition, using the chosen threshold from training folds, and compares each condition with its own cell type's controls. A control-versus-control row checks that vector control and proliferating control aren't themselves separable.

In [ ]:
meta_p = labels.assign(condition=labels.treatment.str.replace(r'\s*\(.*', '', regex=True))
rows = []
for name in ORDER:
    f = pred_df[pred_df.model == name].merge(meta_p[['condition', 'cell_line']], left_on='sample_id', right_index=True)
    for (ct, cond), g in f[f.y == 1].groupby(['cell_line', 'condition']):
        ctrl = f[(f.cell_line == ct) & (f.y == 0)]
        rows.append({'model': name, 'cell_line': ct, 'condition': cond, 'n': len(g), 'auroc_vs_own_controls': sg.roc_auc_score([1] * len(g) + [0] * len(ctrl), list(g.score) + list(ctrl.score))})
by_cond = pd.DataFrame(rows)
by_cond.to_csv(RESULTS / 'dev_auroc_by_condition.csv', index=False)
print(by_cond[by_cond.auroc_vs_own_controls < 1].round(3).to_string(index=False))
print('\nMean AUROC by condition across models and cell types:')
print(by_cond.groupby('condition').auroc_vs_own_controls.agg(['mean', 'min', 'count']).round(3).to_string())
pub_cond = []
for (ct, cond), g in meta_p[meta_p.y == 1].groupby(['cell_line', 'condition']):
    c = meta_p[(meta_p.cell_line == ct) & (meta_p.y == 0)]
    for nm, sc in [('published SenCat', pub_score), ('hUSI raw', husi), ('Hallmark E2F (inverted)', scores['Hallmark E2F (inverted)'])]:
        pub_cond.append({'score': nm, 'cell_line': ct, 'condition': cond, 'auroc': sg.roc_auc_score([1] * len(g) + [0] * len(c), list(sc[g.index]) + list(sc[c.index]))})
pub_cond = pd.DataFrame(pub_cond)
print('\nFixed scores by condition:')
print(pub_cond.pivot_table(index='condition', columns='score', values='auroc').round(3).to_string())
for ct in ['BJ', 'WI-38']:
    i = meta_p.cell_line == ct
    a = meta_p[i & meta_p.condition.isin(['Empty vector'])].index
    b = meta_p[i & meta_p.condition.str.startswith('Control')].index
    print(f'{ct}: published score, proliferating control mean {pub_score[b].mean():.1f} vs empty vector {pub_score[a].mean():.1f}; RasV12 mean {pub_score[meta_p[i & (meta_p.condition == "RasV12")].index].mean():.1f}')

## Calibration, within the development set only
Pooled out-of-fold probabilities for the two tree models and logistic models. They come from different fold models, and calibration on cell types is the question that matters, so these numbers describe held-out cell types. They say nothing about MSC passage samples.

In [ ]:
from sklearn.calibration import calibration_curve
cal_rows = []
for name in ['logreg_l1', 'elastic_net', 'random_forest', 'xgboost']:
    f = pred_df[pred_df.model == name]
    p = 1 / (1 + np.exp(-f.score)) if name in ('logreg_l1', 'elastic_net') else f.score
    frac, mean_pred = calibration_curve(f.y, p, n_bins=5, strategy='quantile')
    cal_rows.append({'model': name, 'brier': sg.brier_score_loss(f.y, p), 'base_rate_brier': f.y.mean() * (1 - f.y.mean()), 'max_bin_gap': float(np.abs(frac - mean_pred).max())})
cal = pd.DataFrame(cal_rows)
cal.to_csv(RESULTS / 'dev_calibration.csv', index=False)
print(cal.round(3).to_string(index=False))

## Freeze: model choice and weights, before notebook 03 reads any MSC score
Applies the rule in the protocol. Three weight sets are fit from SenCat using the authors' pipeline shape, plus the chosen model on all cell types:
- `all14`: all cell types,
- `no_BMMSC`: for cell-type-independent BM-MSC transport,
- `no_BMMSC_prAD`: also drops the adipose-lineage preadipocytes for AD-MSC.

Published weights remain the primary score.

In [ ]:
best_auc = summary.mean_auroc.max()
tied = summary[summary.mean_auroc >= best_auc - 0.01]
chosen = next(m for m in ORDER if m in set(tied.model))
print('Best mean AUROC %.4f; tied within 0.01: %s; chosen by simplicity order: %s' % (best_auc, tied.model.tolist(), chosen))
params = json.loads(fold_df[fold_df.model == chosen].best_params.value_counts().index[0])
print('Most frequent hyperparameters across folds:', params)

def fit_refit_weights(mask, tag):
    c = counts_all.loc[mask]
    filt = sg.author_gene_filter(c)
    n, _, ref = sg.mor_normalize(filt)
    coef, _ = sg.author_pipeline(n, y[mask], groups[mask])
    path = RESULTS / f'frozen_weights_refit_{tag}.csv'
    coef.rename_axis('gene_id').reset_index().to_csv(path, index=False)
    return path, ref

frozen = {}
pub_path = sg.EXT / 'wf-ml-markers-senescence/analysis/transcriptomics_loose5000f_tuned_common_features.csv'
frozen['published_sencat'] = {'path': str(pub_path), 'sha256': sg.sha256(pub_path)}
masks = {'all14': np.ones(len(y), bool), 'no_BMMSC': groups != 'BMMSC', 'no_BMMSC_prAD': ~np.isin(groups, ['BMMSC', 'prAD'])}
for tag, mask in masks.items():
    path, ref = fit_refit_weights(mask, tag)
    frozen[f'refit_{tag}'] = {'path': str(path), 'sha256': sg.sha256(path), 'n_train': int(mask.sum()), 'cell_types': sorted(set(groups[mask]))}
    print(tag, 'trained on', int(mask.sum()), 'samples, weights', pd.read_csv(path).shape[0])

import joblib
pipe, _ = sg.model_grid(xgb_device)[chosen]
model_rows = {}
for tag, mask in masks.items():
    genes, ref = sg.prefilter_train(counts_all.loc[mask])
    Xall = sg.fold_transform(counts_all.loc[mask], genes, ref)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        fitted = sg.clone(pipe).set_params(**params).fit(Xall.to_numpy(), y[mask])
    path = CKPT / f'frozen_{chosen}_{tag}.joblib'
    joblib.dump({'model': fitted, 'genes': genes, 'ref': ref, 'name': chosen, 'params': params, 'tag': tag}, path)
    frozen[f'{chosen}_{tag}'] = {'path': str(path), 'sha256': sg.sha256(path), 'n_train': int(mask.sum())}

freeze = {'frozen_utc': pd.Timestamp.now('UTC').isoformat(), 'chosen_model': chosen, 'params': params, 'selection': {'best_mean_auroc': float(best_auc), 'tied': tied.model.tolist()},
          'protocol_sha256': sg.sha256(REPORTS / 'ANALYSIS_PROTOCOL.md'), 'artifacts': frozen, 'sklearn': sklearn.__version__, 'xgboost': xgboost.__version__, 'seed': sg.SEED}
(RESULTS / 'frozen_model.json').write_text(json.dumps(freeze, indent=1))
print('Frozen. sha256 of frozen_model.json:', sg.sha256(RESULTS / 'frozen_model.json'))

## Figure 1: held-out performance by cell type
Dots are per-cell-type AUROC, with a bootstrap CI over cell types on the mean. MSC-relevant types are marked. Values and their table are in `results/dev_auroc_by_cell_type.csv`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), gridspec_kw={'width_ratios': [1.15, 1]})
ax = axes[0]
disp = {'logreg_l1': 'L1 logistic', 'elastic_net': 'Elastic net', 'linear_svm': 'Linear SVM', 'rbf_svm': 'RBF SVM', 'random_forest': 'Random forest', 'xgboost': 'XGBoost'}
ypos = np.arange(len(ORDER))[::-1]
for yy, name in zip(ypos, ORDER):
    f = fold_df[fold_df.model == name]
    rng = np.random.default_rng(1)
    other = f[~f.held_out.isin(['BMMSC', 'prAD'])]
    ax.scatter(other.auroc, yy + rng.uniform(-0.18, 0.18, len(other)), s=26, color=sg.PAL['muted'], alpha=0.7, linewidths=0, label='Other cell types' if yy == ypos[0] else None, zorder=2)
    for t, c, lab in [('BMMSC', sg.PAL['blue'], 'BMMSC held out'), ('prAD', sg.PAL['orange'], 'Preadipocytes held out')]:
        ax.scatter(f[f.held_out == t].auroc, [yy], s=60, color=c, edgecolors=sg.PAL['surface'], linewidths=1.5, label=lab if yy == ypos[0] else None, zorder=4)
    s = summary[summary.model == name].iloc[0]
    ax.plot([s.ci_low, s.ci_high], [yy - 0.34, yy - 0.34], color=sg.PAL['ink'], lw=2, solid_capstyle='round')
    ax.scatter([s.mean_auroc], [yy - 0.34], s=34, color=sg.PAL['ink'], zorder=5, label='Mean, 95% CI over types' if yy == ypos[0] else None)
ax.set_yticks(ypos, [disp[m] for m in ORDER])
ax.set_xlabel('AUROC on the held-out cell type')
ax.set_title('Nested leave-one-cell-type-out')
ax.set_ylim(-0.75, len(ORDER) - 0.4)
ax.grid(axis='y', visible=False)
ax.legend(loc='lower left', fontsize=8)

ax = axes[1]
show = simple[~simple.score.str.contains('self-consistency')].sort_values('mean_within_type_auroc')
yy = np.arange(len(show))
ax.hlines(yy, show.ci_low, show.ci_high, color=sg.PAL['blue'], lw=2)
ax.scatter(show.mean_within_type_auroc, yy, s=40, color=sg.PAL['blue'], zorder=3, edgecolors=sg.PAL['surface'], linewidths=1.5)
ax.axvline(0.5, color=sg.PAL['muted'], lw=1, ls=(0, (3, 3)))
ax.set_yticks(yy, show.score, fontsize=8.5)
ax.set_xlabel('Mean within-cell-type AUROC, 95% CI')
ax.set_title('Scores that fit nothing on SenCat')
ax.grid(axis='y', visible=False)
fig.tight_layout()
fig.savefig(sg.FIGS / 'fig1_development_performance.png')
plt.show()

## G2 report

In [ ]:
best = summary[summary.model == chosen].iloc[0]
report = f"""# G2: baselines on development data

Status: see the summary below. Data: GSE284666, {len(y)} samples, {int(y.sum())} senescence-induced, {len(np.unique(groups))} cell types. Validation unit: cell type. No donor information exists, so nothing here is donor-independent.

## Published score reproduction [measured]
- Coefficient refit vs published: {repro['overlap']}/{repro['published_features']} shared features, Spearman {repro['spearman_coef']:.3f}, sign agreement {repro['sign_agreement']:.3f}.
- The authors' exact input table is not in GEO, so an exact match was not expected.
- Published score mean within-cell-type AUROC {pub_auc.auroc.mean():.3f}: self-consistency only, because those weights were fit on these samples.

## Feature-selection leakage [measured]
As published {leak.auroc_as_published.mean():.4f} vs clean {leak.auroc_clean.mean():.4f}; mean difference {leak.difference.mean():.4f}.

## Benchmark, nested leave-one-cell-type-out [measured]
{sg.pd.DataFrame(summary).round(3).to_string(index=False)}

Chosen under the protocol rule: {chosen}, params {params}.

## Scores that fit nothing on SenCat [measured]
{simple.round(3).to_string(index=False)}

## Confounding
Variance by PC is in results/dev_confounding_pca.csv. Dose and duration differ by cell type, and paradigms are not crossed with cell type. Replicates sit within a cell line, so per-sample counts overstate independent evidence.
"""
(REPORTS / 'G2_BASELINES.md').write_text(report, encoding='utf-8')
print(report)
outs = [RESULTS / f for f in ['dev_confounding_pca.csv', 'dev_feature_selection_leakage.csv', 'dev_simple_scores.csv', 'dev_simple_scores_per_sample.csv', 'dev_nested_loco_folds.csv', 'dev_nested_loco_predictions.csv',
                              'dev_fold_assignments.csv', 'dev_model_summary.csv', 'dev_auroc_by_cell_type.csv', 'dev_calibration.csv', 'frozen_model.json']] + [sg.FIGS / 'fig1_development_performance.png', REPORTS / 'G2_BASELINES.md']
print(sg.output_manifest(outs).assign(path=lambda d: d.path.str.replace(str(ROOT), '', regex=False))[['path', 'bytes']].to_string())